# Can we fit $\mathcal{H}$ Using DNNs at a Single Kinematic Point Using the KM15/BKM10 Formalism for the Cross-Section?

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Libraries:

In [ ]:
# native libraries
import glob
import gc
import datetime
import yaml
import re
import time
from pathlib import Path

# data analysis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import corner
from scipy.stats import norm
from sklearn.model_selection import train_test_split

# formalism
import gepard as g
from gepard.fits import th_KM15

# BKM10 library stuff:
from bkm10_lib.core import DifferentialCrossSection
from bkm10_lib.inputs import BKM10Inputs
from bkm10_lib.cff_inputs import CFFInputs

# local imports
from unfolded_loss import UnfoldedSimultaneousFitLoss
from simultaneous_fit_dnn_config import bkm10_cross_section

print("[INFO]: Libraries imported!")

### (1.2): Matplotlib Plotting Aesthetics:

In [ ]:
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "savefig.dpi": 300,
    "axes.labelsize": 16,
    "xtick.direction": "in",
    "xtick.major.size": 8.5,
    "xtick.major.width": 1.0,
    "xtick.minor.size": 4.5,
    "xtick.minor.width": 1.0,
    "xtick.minor.visible": True,
    "xtick.top": True,
    "ytick.direction": "in",
    "ytick.major.size": 8.5,
    "ytick.major.width": 1.0,
    "ytick.minor.size": 4.5,
    "ytick.minor.width": 1.0,
    "ytick.minor.visible": True,
    "ytick.right": True,
    "xtick.labelsize": 15.0,
    "ytick.labelsize": 15.0,
})

print("[INFO]: Customized Matplotlib settings!")

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: tensorflow version: {tf.__version__}")
print(f"[INFO]: gepard version: {g.__version__}")
print(f"[INFO]: corner version: {corner.__version__}")

### (1.4): TensorFlow Configurations:

In [ ]:
cpus = tf.config.list_physical_devices('CPU')
gpus = tf.config.list_physical_devices('GPU')

print(f"[INFO]: Number of CPUs Available: {len(cpus)}")
print(f"[INFO]: Number of GPUs Available: {len(gpus)}")
print(f"[INFO]: The available devices are: {tf.config.list_physical_devices()}")
print(f"[INFO]: Is CUDA-built? {tf.test.is_built_with_cuda()}")

## (2): Configuring 

### (2.1): Reading the YAML file:

In [ ]:
with open(
    "closure_test_config.yml",
    "r",
    encoding = "utf-8") as file:
    config = yaml.safe_load(file)

### (2.2): Obtaining the `SCRATCH_PATH`:

In [ ]:
SCRATCH_PATH = config["scratch_path"]
print(f"[INFO]: Computed scratch path: {SCRATCH_PATH}")

### (2.3): Obtaining versioning info:

In [ ]:
MAJOR_NUMBER = config["versioning"]["major"]
MINOR_NUMBER = config["versioning"]["minor"]
MAJOR_MINOR_NUMBER = f"{MAJOR_NUMBER}_{MINOR_NUMBER}"

print(f"[INFO]: Recieved major version number: {MAJOR_NUMBER}")
print(f"[INFO]: Recieved minor version number: {MINOR_NUMBER}")
print(f"[INFO]: Recieved total version number: {MAJOR_MINOR_NUMBER}")

### (2.4): Constructing dynamic `Path` objects:
1. `data/`
2. `plots/`
3. `replicas/`
4. `learning_curves/`

In [ ]:
version_directory = Path(SCRATCH_PATH) / f"version_{MAJOR_MINOR_NUMBER}"

data_directory = version_directory / "data"
plots_directory = version_directory / "plots"
replica_directory = version_directory / "replicas"
learning_curves_directory = version_directory / "learning_curves"

data_directory.mkdir(parents = True, exist_ok = True)
plots_directory.mkdir(parents = True, exist_ok = True)
learning_curves_directory.mkdir(parents = True, exist_ok = True)
replica_directory.mkdir(parents = True, exist_ok = True)

### (2.5): Reading in DNN hyperparameter information:

In [ ]:
NUMBER_OF_EPOCHS = config["dnn_config"]["epochs"]
NUMBER_OF_REPLICAS = config["dnn_config"]["replicas"]
BATCH_SIZE = config["dnn_config"]["batch_size"]
LEARNING_RATE = config["dnn_config"]["adam_learning_rate"]

_NUMBER_OF_NODES_LAYER_1 = 20
_NUMBER_OF_NODES_LAYER_2 = 20
_NUMBER_OF_NODES_LAYER_3 = 20
_NUMBER_OF_NODES_LAYER_4 = 20

print(f"[INFO]: Received number of epochs (per replica): {NUMBER_OF_EPOCHS}")
print(f"[INFO]: Received number of replicas: {NUMBER_OF_REPLICAS}")
print(f"[INFO]: Received batch size: {BATCH_SIZE}")
print(f"[INFO]: Received (Adam) learning rate value: {LEARNING_RATE}")

### (2.6): Constructing boolean flags telling us which CFFs are fit by the DNN or not:

In [ ]:
IS_CFF_REAL_H_FREE = config["cff_config"]["enable_cff_real_h"]
print(f"[INFO]: Is Re[H] free to fit? {IS_CFF_REAL_H_FREE}")
IS_CFF_IMAG_H_FREE = config["cff_config"]["enable_cff_imag_h"]
print(f"[INFO]: Is Im[H] free to fit? {IS_CFF_IMAG_H_FREE}")
IS_CFF_REAL_HT_FREE = config["cff_config"]["enable_cff_real_ht"]
print(f"[INFO]: Is Re[Ht] free to fit? {IS_CFF_REAL_HT_FREE}")
IS_CFF_IMAG_HT_FREE = config["cff_config"]["enable_cff_imag_ht"]
print(f"[INFO]: Is Im[Ht] free to fit? {IS_CFF_IMAG_HT_FREE}")
IS_CFF_REAL_E_FREE = config["cff_config"]["enable_cff_real_e"]
print(f"[INFO]: Is Re[E] free to fit? {IS_CFF_REAL_E_FREE}")
IS_CFF_IMAG_E_FREE = config["cff_config"]["enable_cff_imag_e"]
print(f"[INFO]: Is Im[E] free to fit? {IS_CFF_IMAG_E_FREE}")
IS_CFF_REAL_ET_FREE = config["cff_config"]["enable_cff_real_et"]
print(f"[INFO]: Is Re[Et] free to fit? {IS_CFF_REAL_ET_FREE}")
IS_CFF_IMAG_ET_FREE = config["cff_config"]["enable_cff_imag_et"]
print(f"[INFO]: Is Im[Et] free to fit? {IS_CFF_IMAG_ET_FREE}")

### (2.7): Constructing boolean flags to tell the program to set various CFFs equal to $0$:

In [ ]:
IS_CFF_REAL_H_QUENCHED = config["cff_quenching"]["quench_cff_real_h"]
print(f"[INFO]: Is Re[H] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_REAL_H_QUENCHED}")
IS_CFF_IMAG_H_QUENCHED = config["cff_quenching"]["quench_cff_imag_h"]
print(f"[INFO]: Is Im[H] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_IMAG_H_QUENCHED}")
IS_CFF_REAL_HT_QUENCHED = config["cff_quenching"]["quench_cff_real_ht"]
print(f"[INFO]: Is Re[Ht] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_REAL_HT_QUENCHED}")
IS_CFF_IMAG_HT_QUENCHED = config["cff_quenching"]["quench_cff_imag_ht"]
print(f"[INFO]: Is Im[Ht] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_IMAG_HT_QUENCHED}")
IS_CFF_REAL_E_QUENCHED = config["cff_quenching"]["quench_cff_real_e"]
print(f"[INFO]: Is Re[E] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_REAL_E_QUENCHED}")
IS_CFF_IMAG_E_QUENCHED = config["cff_quenching"]["quench_cff_imag_e"]
print(f"[INFO]: Is Im[E] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_IMAG_E_QUENCHED}")
IS_CFF_REAL_ET_QUENCHED = config["cff_quenching"]["quench_cff_real_et"]
print(f"[INFO]: Is Re[Et] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_REAL_ET_QUENCHED}")
IS_CFF_IMAG_ET_QUENCHED = config["cff_quenching"]["quench_cff_imag_et"]
print(f"[INFO]: Is Im[Et] quenched (i.e. fixed to 0 and not fit here)? {IS_CFF_IMAG_ET_QUENCHED}")

### (2.8): Constructing a "master list" that defines globally the CFF fitting configuration:

In [ ]:
CFF_CONFIG = {
    "cff_h_real": (
        config["cff_config"]["enable_cff_real_h"],
        config["cff_quenching"]["quench_cff_real_h"],
    ),
    "cff_h_imag": (
        config["cff_config"]["enable_cff_imag_h"],
        config["cff_quenching"]["quench_cff_imag_h"],
    ),
    "cff_ht_real": (
        config["cff_config"]["enable_cff_real_ht"],
        config["cff_quenching"]["quench_cff_real_ht"],
    ),
    "cff_ht_imag": (
        config["cff_config"]["enable_cff_imag_ht"],
        config["cff_quenching"]["quench_cff_imag_ht"],
    ),
    "cff_e_real": (
        config["cff_config"]["enable_cff_real_e"],
        config["cff_quenching"]["quench_cff_real_e"],
    ),
    "cff_e_imag": (
        config["cff_config"]["enable_cff_imag_e"],
        config["cff_quenching"]["quench_cff_imag_e"],
    ),
    "cff_et_real": (
        config["cff_config"]["enable_cff_real_et"],
        config["cff_quenching"]["quench_cff_real_et"],
    ),
    "cff_et_imag": (
        config["cff_config"]["enable_cff_imag_et"],
        config["cff_quenching"]["quench_cff_imag_et"],
    ),
}

# validating the configuration settings (checking consistency):
for cff_name, (is_free, is_quenched) in CFF_CONFIG.items():

    if is_free and is_quenched:
        raise RuntimeError(
            "[ERROR]: I discovered that your configuration file specifies that "
            f"{cff_name} to be both free and quenched. Check it for discrepancies."
        )

CFF_ORDER = (
    "cff_h_real",
    "cff_h_imag",
    "cff_ht_real",
    "cff_ht_imag",
    "cff_e_real",
    "cff_e_imag",
    "cff_et_real",
    "cff_et_imag",
)

FREE_CFF_NAMES = [
    cff_name for cff_name in CFF_ORDER if CFF_CONFIG[cff_name][0] and not CFF_CONFIG[cff_name][1]
]

print(f"[INFO]: The names of the free CFFs we're fitting are {FREE_CFF_NAMES}")

# dynamically compute the number of free CFFs (parameters):
NUMBER_OF_FREE_CFFS = len(FREE_CFF_NAMES)

print(f"[INFO]: The total number of free CFFs we're fitting is {NUMBER_OF_FREE_CFFS}")

if NUMBER_OF_FREE_CFFS == 0:
    raise RuntimeError(
        "[ERROR]: What the hell are you doing? Quenching all 8 CFFS? Sounds like a "
        "boring fit to me!"
    )

### (2.9): Now, we determine what observables are available for fitting here:

In [ ]:
IS_UNP_BEAM_UNP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_unp_beam_unp_target_xsec"]
print(f"[INFO]: Are we fitting sigma(0, 0): {IS_UNP_BEAM_UNP_TARGET_XSEC_INCLUDED}")
IS_PLUS_BEAM_UNP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_plus_beam_unp_target_xsec"]
print(f"[INFO]: Are we fitting sigma(+1, 0): {IS_PLUS_BEAM_UNP_TARGET_XSEC_INCLUDED}")
IS_MINUS_BEAM_UNP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_minus_beam_unp_target_xsec"]
print(f"[INFO]: Are we fitting sigma(-1, 0): {IS_MINUS_BEAM_UNP_TARGET_XSEC_INCLUDED}")

IS_UNP_BEAM_PLUS_TARGET_XSEC_INCLUDED = (config["observable_config"]["enable_unp_beam_plus_target_xsec"])
print(f"[INFO]: Are we fitting sigma(0, +1/2): {IS_UNP_BEAM_PLUS_TARGET_XSEC_INCLUDED}")
IS_PLUS_BEAM_PLUS_TARGET_XSEC_INCLUDED = (config["observable_config"]["enable_plus_beam_plus_target_xsec"])
print(f"[INFO]: Are we fitting sigma(-1, +1/2): {IS_PLUS_BEAM_PLUS_TARGET_XSEC_INCLUDED}")
IS_MINUS_BEAM_PLUS_TARGET_XSEC_INCLUDED = (config["observable_config"]["enable_minus_beam_plus_target_xsec"])
print(f"[INFO]: Are we fitting sigma(-1, +1/2): {IS_MINUS_BEAM_PLUS_TARGET_XSEC_INCLUDED}")

IS_UNP_BEAM_MINUS_TARGET_XSEC_INCLUDED = (config["observable_config"]["enable_unp_beam_minus_target_xsec"])
print(f"[INFO]: Are we fitting sigma(0, -1/2): {IS_UNP_BEAM_MINUS_TARGET_XSEC_INCLUDED}")
IS_PLUS_BEAM_MINUS_TARGET_XSEC_INCLUDED = (config["observable_config"]["enable_plus_beam_minus_target_xsec"])
print(f"[INFO]: Are we fitting sigma(-1, -1/2): {IS_PLUS_BEAM_MINUS_TARGET_XSEC_INCLUDED}")
IS_MINUS_BEAM_MINUS_TARGET_XSEC_INCLUDED = (config["observable_config"]["enable_minus_beam_minus_target_xsec"])
print(f"[INFO]: Are we fitting sigma(-1, -1/2): {IS_MINUS_BEAM_MINUS_TARGET_XSEC_INCLUDED}")

IS_UNP_TARGET_BSA_INCLUDED = config["observable_config"]["enable_unp_target_bsa"]
print(f"[INFO]: Are we fitting BSA(0): {IS_UNP_TARGET_BSA_INCLUDED}")
IS_PLUS_TARGET_BSA_INCLUDED = config["observable_config"]["enable_plus_target_bsa"]
print(f"[INFO]: Are we fitting BSA(+1/2): {IS_PLUS_TARGET_BSA_INCLUDED}")
IS_MINUS_TARGET_BSA_INCLUDED = config["observable_config"]["enable_minus_target_bsa"]
print(f"[INFO]: Are we fitting BSA(-1/2): {IS_MINUS_TARGET_BSA_INCLUDED}")

IS_UNP_BEAM_TSA_INCLUDED = config["observable_config"]["enable_unp_beam_tsa"]
print(f"[INFO]: Are we fitting TSA(0): {IS_UNP_BEAM_TSA_INCLUDED}")
IS_PLUS_BEAM_TSA_INCLUDED = config["observable_config"]["enable_plus_beam_tsa"]
print(f"[INFO]: Are we fitting TSA(+1): {IS_PLUS_BEAM_TSA_INCLUDED}")
IS_MINUS_BEAM_TSA_INCLUDED = config["observable_config"]["enable_minus_beam_tsa"]
print(f"[INFO]: Are we fitting TSA(-1): {IS_MINUS_BEAM_TSA_INCLUDED}")

IS_DSA_INCLUDED = config["observable_config"]["enable_dsa"]
print(f"[INFO]: Are we fitting BSA: {IS_DSA_INCLUDED}")

### (2.10): Constructing a list of `enabled_observables`, i.e. the observables that will be used to compute losses:

In [ ]:
enabled_observables = []

if IS_UNP_BEAM_UNP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("unp_beam_unp_target_xsec")

if IS_PLUS_BEAM_UNP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("plus_beam_unp_target_xsec")

if IS_MINUS_BEAM_UNP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("minus_beam_unp_target_xsec")

if IS_UNP_BEAM_PLUS_TARGET_XSEC_INCLUDED:
    enabled_observables.append("unp_beam_plus_target_xsec")

if IS_PLUS_BEAM_PLUS_TARGET_XSEC_INCLUDED:
    enabled_observables.append("plus_beam_plus_target_xsec")

if IS_MINUS_BEAM_PLUS_TARGET_XSEC_INCLUDED:
    enabled_observables.append("minus_beam_plus_target_xsec")

if IS_UNP_BEAM_MINUS_TARGET_XSEC_INCLUDED:
    enabled_observables.append("unp_beam_minus_target_xsec")

if IS_PLUS_BEAM_MINUS_TARGET_XSEC_INCLUDED:
    enabled_observables.append("plus_beam_minus_target_xsec")

if IS_MINUS_BEAM_MINUS_TARGET_XSEC_INCLUDED:
    enabled_observables.append("minus_beam_minus_target_xsec")

if IS_UNP_TARGET_BSA_INCLUDED:
    enabled_observables.append("unp_target_bsa")

if IS_PLUS_TARGET_BSA_INCLUDED:
    enabled_observables.append("plus_target_bsa")

if IS_MINUS_TARGET_BSA_INCLUDED:
    enabled_observables.append("minus_target_bsa")

if IS_UNP_BEAM_TSA_INCLUDED:
    enabled_observables.append("unp_beam_tsa")

if IS_PLUS_BEAM_TSA_INCLUDED:
    enabled_observables.append("plus_beam_tsa")

if IS_MINUS_BEAM_TSA_INCLUDED:
    enabled_observables.append("minus_beam_tsa")

if IS_DSA_INCLUDED:
    enabled_observables.append("dsa")

print(f"[INFO]: Enabled observables that will go into the fit are {enabled_observables}")

### (2.11): Determining how to best distribute the "weights" present in the simultaneous loss function: $L = \sum_{i = 1}^{N} w_{i} L_{i}$.

The weights are the $w_{i}$ s, and they need to sum to $1$.

In [ ]:
OBSERVABLE_WEIGHTS = []

if len(enabled_observables) == 0:
    raise ValueError("[ERROR]: I didn't see that you enabled any observables to fit...")

if config["observable_config"]["equally_weighted_observables"]:
    observable_weight = 1.0 / len(enabled_observables)
    OBSERVABLE_WEIGHTS = [observable_weight] * len(enabled_observables)
    assert np.isclose(sum(OBSERVABLE_WEIGHTS), 1.0), "[ASSERT]: Something strange happened..."

else:
    raise NotImplementedError(
        "[ERROR]: Sorry! I haven't yet implemented inequal weighting of observables in this "
        "workflow... Stay tuned!"
    )

print(f"[INFO]: Dynamically computed the weights to be {OBSERVABLE_WEIGHTS}")

## (3): Fixing the Physics Data for the Fitting Procedure:

### (3.1): **Defining the Kinematic Point**:

In [ ]:
FIXED_K = 5.750
FIXED_XB = 0.2015384615384615
FIXED_T = -0.2407692307692307
FIXED_Q_SQUARED = 1.6599999999999997

print(f"[INFO]: Received k = {FIXED_K} GeV")
print(f"[INFO]: Received xB = {FIXED_XB}")
print(f"[INFO]: Received t = {FIXED_T} GeV^2")
print(f"[INFO]: Received Q^2 = {FIXED_Q_SQUARED} GeV^2")

### (3.2): Fixing the $\lambda$ and $\Lambda$ for the cross-section fitting:

In [ ]:
TEST_LEPTON_HELICITY = 0.0
TEST_TARGET_POLARIZATION = 0.0

print(f"[INFO]: Lepton beam helicity is = {TEST_LEPTON_HELICITY}")
print(f"[INFO]: Target polarization is = {TEST_TARGET_POLARIZATION}")

### (3.3): Defining the range and total points across $\phi$:

In [ ]:
STARTING_PHI_VALUE_IN_DEGREES = config["data_config"]["start_value_of_phi_in_degrees"]
ENDING_PHI_VALUE_IN_DEGREES = config["data_config"]["end_value_of_phi_in_degrees"]
NUMBER_OF_PHI_POINTS = config["data_config"]["number_of_phi_points"] + 1

phi_array_in_degrees = np.linspace(
    start = STARTING_PHI_VALUE_IN_DEGREES,
    stop = ENDING_PHI_VALUE_IN_DEGREES,
    num = NUMBER_OF_PHI_POINTS)

phi_array_in_radians = [np.radians(degree_value) for degree_value in phi_array_in_degrees]

print(f"[INFO]: New list of {len(phi_array_in_radians)} of azimuthal angles from {STARTING_PHI_VALUE_IN_DEGREES} degrees to {ENDING_PHI_VALUE_IN_DEGREES} degrees.")

## (4): Generating the "Ground Truth" CFF Settings:

### (4.1): Use the KM15 Model of to Generate "Ground Truth" Values:

Uses `gepard`'s `DataPoint()` class to define the kinematic setting.

In [ ]:
try:
    # [NOTE]: We actually don't need to be super accurate here because we ONLY use
    # this class to evaluate the CFFs later!
    test_datapoints = [
        g.DataPoint(
            xB = FIXED_XB,
            t = FIXED_T,
            Q2 = FIXED_Q_SQUARED,
            phi = fixed_phi,
            process = "ep2epgamma",
            exptype = 'fixed target',
            in1energy = FIXED_K,
            in1charge = -1,
            in1polarization = +1,
            in1units = 'rad',
            observable = 'XS',
            fname = 'Trento') for fixed_phi in phi_array_in_radians]
except ZeroDivisionError:
    print(
        f"[ERROR]: Kinematic setting k = {FIXED_K}, xb = {FIXED_XB}, "
        f"t = {FIXED_T}, Q^2 = {FIXED_Q_SQUARED} unphysical according to gepard."
        )

#### (4.1.1): A basic assert:

In [ ]:
assert (
    len(test_datapoints) == len(phi_array_in_radians)
), "[ASSERT]: I think your Gepard + list comprehension did something strange."

### (4.2): Here, we actually obtain all the CFFs using KM15 for the Datapoints:

In [ ]:
# here we compute the actual CFFs!
real_h_values = np.array([th_KM15.ReH(datapoint) for datapoint in test_datapoints])
imag_h_values = np.array([th_KM15.ImH(datapoint) for datapoint in test_datapoints])
real_e_values = np.array([th_KM15.ReE(datapoint) for datapoint in test_datapoints])
imag_e_values = np.array([th_KM15.ImE(datapoint) for datapoint in test_datapoints])
real_ht_values = np.array([th_KM15.ReHt(datapoint) for datapoint in test_datapoints])
imag_ht_values = np.array([th_KM15.ImHt(datapoint) for datapoint in test_datapoints])
real_et_values = np.array([th_KM15.ReEt(datapoint) for datapoint in test_datapoints])
imag_et_values = np.array([th_KM15.ImEt(datapoint) for datapoint in test_datapoints])

#### (4.2.1): Check out the Length of the CFF Data Array:

These arrays should ALSO be of exactly the same length --- For every phi, we have a kinematic point, and every kinematic point will have a CFF value.

In [ ]:
assert (
    len(real_h_values) == len(test_datapoints)
), "[ASSERT]: Your Re[H] values are not as many as the datapoints you have..."
assert (
    len(imag_h_values) == len(test_datapoints)
), "[ASSERT]: Your Im[H] values are not as many as the datapoints you have..."
assert (
    len(real_e_values) == len(test_datapoints)
), "[ASSERT]: Your Re[E] values are not as many as the datapoints you have..."
assert (
    len(imag_e_values) == len(test_datapoints)
), "[ASSERT]: Your Im[E] values are not as many as the datapoints you have..."
assert (
    len(real_ht_values) == len(test_datapoints)
), "[ASSERT]: Your Re[Ht] values are not as many as the datapoints you have..."
assert (
    len(imag_ht_values) == len(test_datapoints)
), "[ASSERT]: Your Im[Ht] values are not as many as the datapoints you have..."
assert (
    len(real_et_values) == len(test_datapoints)
), "[ASSERT]: Your Re[Et] values are not as many as the datapoints you have..."
assert (
    len(imag_et_values) == len(test_datapoints)
), "[ASSERT]: Your Im[Et] values are not as many as the datapoints you have..."

### (4.3): Globally define the KM15 CFF values:

In [ ]:
# here we actually set the KM15 values to 0:
CFF_REAL_H_KM15 = real_h_values[0] if IS_CFF_REAL_H_FREE else 0.0
print(f"[INFO]: Setting Re[H] = {CFF_REAL_H_KM15}")
CFF_IMAG_H_KM15 = imag_h_values[0] if IS_CFF_IMAG_H_FREE else 0.0
print(f"[INFO]: Setting Im[H] = {CFF_IMAG_H_KM15}")
CFF_REAL_HT_KM15 = real_ht_values[0] if IS_CFF_REAL_HT_FREE else 0.0
print(f"[INFO]: Setting Re[Ht] = {CFF_REAL_HT_KM15}")
CFF_IMAG_HT_KM15 = imag_ht_values[0] if IS_CFF_IMAG_HT_FREE else 0.0
print(f"[INFO]: Setting Im[Ht] = {CFF_IMAG_HT_KM15}")
CFF_REAL_E_KM15 = real_e_values[0] if IS_CFF_REAL_E_FREE else 0.0
print(f"[INFO]: Setting Re[E] = {CFF_REAL_E_KM15}")
CFF_IMAG_E_KM15 = imag_e_values[0] if IS_CFF_IMAG_E_FREE else 0.0
print(f"[INFO]: Setting Im[E] = {CFF_IMAG_E_KM15}")
CFF_REAL_ET_KM15 = real_et_values[0] if IS_CFF_REAL_ET_FREE else 0.0
print(f"[INFO]: Setting Re[Et] = {CFF_REAL_ET_KM15}")
CFF_IMAG_ET_KM15 = imag_et_values[0] if IS_CFF_IMAG_ET_FREE else 0.0
print(f"[INFO]: Setting Im[Et] = {CFF_IMAG_ET_KM15}")

CFF_H_KM15 = complex(CFF_REAL_H_KM15, CFF_IMAG_H_KM15)
CFF_H_TILDE_KM15 = complex(CFF_REAL_HT_KM15, CFF_IMAG_HT_KM15)
CFF_E_KM15 = complex(CFF_REAL_E_KM15, CFF_IMAG_E_KM15)
CFF_E_TILDE_KM15 = complex(CFF_REAL_ET_KM15, CFF_IMAG_ET_KM15)

### (4.4): Preparing for future "plot strings"

#### (4.4.1): The kinematic setting plot string:

In [ ]:
this_kinematic_set_title_string = (
    rf"$k = {FIXED_K:.3f}$ GeV, "
    rf"$x_B = {FIXED_XB:.3f}$, "
    rf"$t = {FIXED_T:.3f}$ GeV$^2$, "
    rf"$Q^2 = {FIXED_Q_SQUARED:.3f}$ GeV$^2$"
)

print(this_kinematic_set_title_string)

#### (4.4.2): The KM15 CFF setting plot string:

In [ ]:
km15_cff_string = (
    rf"$\mathcal{{H}} = {CFF_H_KM15:.3f}$, "
    rf"$\mathcal{{E}} = {CFF_E_KM15:.3f}$, "
    rf"$\widetilde{{\mathcal{{H}}}} = {CFF_H_TILDE_KM15:.3f}$, "
    rf"$\widetilde{{\mathcal{{E}}}} = {CFF_E_TILDE_KM15:.3f}$ "
)

print(km15_cff_string)

### (4.5): Use the [BKM10 Library](https://pypi.org/project/bkm10/) to Predict *all* of the Observables:

#### (4.5.2): First, we initialize a `bkm10` "computational object":

In [ ]:
km15_cff_computational_hub = DifferentialCrossSection(
    configuration = {
        "kinematics": BKM10Inputs(
            lab_kinematics_k = FIXED_K,
            squared_Q_momentum_transfer = FIXED_Q_SQUARED,
            x_Bjorken = FIXED_XB,
            squared_hadronic_momentum_transfer_t = FIXED_T),
        "cff_inputs": CFFInputs(
            compton_form_factor_h = CFF_H_KM15,
            compton_form_factor_h_tilde = CFF_H_TILDE_KM15,
            compton_form_factor_e = CFF_E_KM15,
            compton_form_factor_e_tilde = CFF_E_TILDE_KM15),
        "using_ww": True
    },
    verbose = False,
    debugging = False)

##### (4.5.2.1): $d^{4}\sigma^{UU}$ using **KM15** CFFs:

In [ ]:
bkm10_unp_beam_unp_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = 0.0,
    target_polarization = 0.0).real

print("[INFO]: Used bkm10 library to compute sigma(0, 0)")

##### (4.5.2.2): $d^{4}\sigma^{+U}$ using **KM15** CFFs:

In [ ]:
bkm10_plus_beam_unp_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = +1.0,
    target_polarization = 0.0).real

print("[INFO]: Used bkm10 library to compute sigma(+1, 0)")

##### (4.5.2.3): $d^{4}\sigma^{-U}$ using **KM15** CFFs:

In [ ]:
bkm10_minus_beam_unp_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = -1.0,
    target_polarization = 0.0).real

print("[INFO]: Used bkm10 library to compute sigma(-1, 0)")

##### (4.5.2.4): $d^{4}\sigma^{U+}$ using **KM15** CFFs:

In [ ]:
bkm10_unp_beam_plus_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = 0.0,
    target_polarization = +0.5).real

print("[INFO]: Used bkm10 library to compute sigma(0, +0.5)")

##### (4.5.2.5): $d^{4}\sigma^{++}$ using **KM15** CFFs:

In [ ]:
bkm10_plus_beam_plus_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = +1.0,
    target_polarization = +0.5).real

print("[INFO]: Used bkm10 library to compute sigma(+1, +0.5)")

##### (4.5.2.6): $d^{4}\sigma^{-+}$ using **KM15** CFFs:

In [ ]:
bkm10_minus_beam_plus_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = -1.0,
    target_polarization = +0.5).real

print("[INFO]: Used bkm10 library to compute sigma(-1, +0.5)")

##### (4.5.2.7): $d^{4}\sigma^{U-}$ using **KM15** CFFs:

In [ ]:
bkm10_unp_beam_minus_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = 0.0,
    target_polarization = +0.5).real

print("[INFO]: Used bkm10 library to compute sigma(0, +0.5)")

##### (4.5.2.8): $d^{4}\sigma^{+-}$ using **KM15** CFFs:

In [ ]:
bkm10_plus_beam_minus_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = +1.0,
    target_polarization = -0.5).real

print("[INFO]: Used bkm10 library to compute sigma(0, +0.5)")

##### (4.5.2.7): $d^{4}\sigma^{--}$ using **KM15** CFFs:

In [ ]:
bkm10_minus_beam_minus_target_km15 = km15_cff_computational_hub.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = -1.0,
    target_polarization = -0.5).real

print("[INFO]: Used bkm10 library to compute sigma(0, +0.5)")

##### (4.5.2.7): $\text{BSA}\left( \Lambda = 0 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_bsa_unp_target_km15 = km15_cff_computational_hub.compute_bsa(
    phi_array_in_radians,
    target_polarization = 0.0).real

print("[INFO]: Used bkm10 library to compute BSA(0)")

##### (4.5.2.8): $\text{BSA}\left( \Lambda = +1/2 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_bsa_plus_target_km15 = km15_cff_computational_hub.compute_bsa(
    phi_array_in_radians,
    target_polarization = +0.5).real

print("[INFO]: Used bkm10 library to compute BSA(+0.5)")

##### (4.5.2.9): $\text{BSA}\left( \Lambda = -1/2 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_bsa_minus_target_km15 = km15_cff_computational_hub.compute_bsa(
    phi_array_in_radians,
    target_polarization = -0.5).real

print("[INFO]: Used bkm10 library to compute BSA(-0.5)")

##### (4.5.2.10): $\text{TSA}\left( \lambda = 0 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_unp_beam_tsa_km15 = km15_cff_computational_hub.compute_tsa(
    phi_array_in_radians,
    lepton_polarization = 0.0).real

print("[INFO]: Used bkm10 library to compute TSA(0)")

##### (4.5.2.11): $\text{TSA}\left( \lambda = +1 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_plus_beam_tsa_km15 = km15_cff_computational_hub.compute_tsa(
    phi_array_in_radians,
    lepton_polarization = +1.0).real

print("[INFO]: Used bkm10 library to compute TSA(+1)")

##### (4.5.2.12): $\text{TSA}\left( \lambda = -1 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_minus_beam_tsa_km15 = km15_cff_computational_hub.compute_tsa(
    phi_array_in_radians,
    lepton_polarization = -1.0).real

print("[INFO]: Used bkm10 library to compute TSA(-1)")

##### (4.5.2.13): $\text{DSA}$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_dsa_km15 = km15_cff_computational_hub.compute_dsa(phi_array_in_radians).real

print("[INFO]: Used bkm10 library to compute DSA")

### (4.6): Let's quickly make plots of the BKM10-predicted observables:

#### (4.6.1): A function that plots all the BKM10-predicted observables:

In [ ]:
def plot_bkm10_observable(
    x_data,
    y_data,
    y_label,
    title,
    observable_label,
    figure_filename):
    
    figure, axis = plt.subplots(
        ncols = 1,
        nrows = 1,
        figsize = (10, 10))

    axis.scatter(
        x_data,
        y_data,
        s = 4.0,
        color = "blue",
        label = observable_label)

    axis.set_xlabel(
        r"$\phi$ [radians]",
        fontsize = 20.)
    
    axis.set_ylabel(
        y_label,
        fontsize = 20.)
    
    axis.set_title(
        title,
        fontsize = 16.)

    axis.legend(
        fontsize = 20.)
    
    axis.grid(
        visible = True,
        alpha = 0.35)

    for extension in ["png", "eps"]:
        figure.savefig(
            plots_directory /
            f"{figure_filename}.{extension}",
            facecolor = "white"
        )

    plt.close(figure)

#### (4.5.2): We now make the plots in this block:

In [ ]:
observables_to_plot = {
    "unp_beam_unp_target_xsec": {
        "data": bkm10_unp_beam_unp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{UU}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_unp_beam_unp_target_prediction",
    },
    "plus_beam_unp_target_xsec": {
        "data": bkm10_plus_beam_unp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{+U}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_plus_beam_unp_target_prediction",
    },
    "minus_beam_unp_target_xsec": {
        "data": bkm10_minus_beam_unp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{-U}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_minus_beam_unp_target_prediction",
    },
    "unp_beam_plus_target_xsec": {
        "data": bkm10_unp_beam_plus_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{U+}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_unp_beam_plus_target_prediction",
    },
    "plus_beam_plus_target_xsec": {
        "data": bkm10_plus_beam_plus_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{++}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_plus_beam_plus_target_prediction",
    },
    "minus_beam_plus_target_xsec": {
        "data": bkm10_minus_beam_plus_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{-+}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_minus_beam_plus_target_prediction",
    },
    "unp_beam_minus_target_xsec": {
        "data": bkm10_unp_beam_minus_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{U-}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_unp_beam_minus_target_prediction",
    },
    "plus_beam_minus_target_xsec": {
        "data": bkm10_plus_beam_minus_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{+-}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_plus_beam_minus_target_prediction",
    },
    "minus_beam_minus_target_xsec": {
        "data": bkm10_minus_beam_minus_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{--}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_minus_beam_minus_target_prediction",
    },
    "unp_target_bsa": {
        "data": bkm10_bsa_unp_target_km15,
        "label": r"BKM10 $\textrm{BSA}(\Lambda = 0)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{BSA}(\Lambda = 0)$ [unitless]",
        "filename": "bkm10_bsa_unp_target_prediction",
    },
    "plus_lp_target_bsa": {
        "data": bkm10_bsa_plus_target_km15,
        "label": r"BKM10 $\textrm{BSA}(\Lambda = +1/2)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{BSA}(\Lambda = +1/2)$ [unitless]",
        "filename": "bkm10_bsa_plus_lp_target_prediction",
    },
    "minus_lp_target_bsa": {
        "data": bkm10_bsa_minus_target_km15,
        "label": r"BKM10 $\textrm{BSA}(\Lambda = -1/2)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{BSA}(\Lambda = -1/2)$ [unitless]",
        "filename": "bkm10_bsa_minus_lp_target_prediction",
    },
    "unp_beam_tsa": {
        "data": bkm10_unp_beam_tsa_km15,
        "label": r"BKM10 $\textrm{TSA}(\lambda = 0)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{TSA}(\lambda = 0)$ [unitless]",
        "filename": "bkm10_tsa_unp_beam_prediction",
    },
    "plus_beam_tsa": {
        "data": bkm10_plus_beam_tsa_km15,
        "label": r"BKM10 $\textrm{TSA}(\lambda = +1)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{TSA}(\lambda = +1)$ [unitless]",
        "filename": "bkm10_tsa_plus_beam_prediction",
    },
    "minus_beam_tsa": {
        "data": bkm10_minus_beam_tsa_km15,
        "label": r"BKM10 $\textrm{TSA}(\lambda = -1)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{TSA}(\lambda = -1)$ [unitless]",
        "filename": "bkm10_tsa_minus_beam_prediction",
    },
    "dsa": {
        "data": bkm10_dsa_km15,
        "label": r"BKM10 $\textrm{DSA}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{DSA}$ [unitless]",
        "filename": "bkm10_dsa_prediction",
    },
}

for observable_name, observable in observables_to_plot.items():

    print(f"[INFO]: Now making a plot showing BKM10-predicted {observable_name}")

    plot_bkm10_observable(
        phi_array_in_radians,
        observable["data"],
        y_label = observable["ylabel"],
        title = (
            rf"{observable_name} vs. $\phi$, "
            f"{this_kinematic_set_title_string}"    
            "\n"
            f"(KM15): {km15_cff_string}"
        ),
        observable_label = observable["label"],
        figure_filename = observable['filename']
    )

## (4): DNN Model Definitions:

We need to go through all of this section in order to get TensorFlow working with the BKM10 custom computation for the cross-section and BSA.

### (4.1): Static Quantities Used in Loss Computation:

In [ ]:
_MASS_OF_PROTON_IN_GEV = 0.93827208816
_ELECTRIC_FORM_FACTOR_CONSTANT = 0.710649
_PROTON_MAGNETIC_MOMENT = 2.79284734463

def compute_fe(t):
    return 1./ (1. - (t / _ELECTRIC_FORM_FACTOR_CONSTANT))**2

def compute_fg(fe):
    return _PROTON_MAGNETIC_MOMENT * fe

def compute_f2(t, fe, fg):
    tau = -1. * t/ (4. * _MASS_OF_PROTON_IN_GEV**2)
    numerator = fg - fe
    denominator = 1. + tau
    return numerator / denominator

def compute_f1(fg, f2):
    return fg - f2

def compute_epsilon(xb, q_squared):
    return 2. * xb * _MASS_OF_PROTON_IN_GEV / tf.sqrt(q_squared)

def compute_y(k_beam, q_squared, ep):
    return tf.sqrt(q_squared) / (ep * k_beam)

def compute_skewness(xb, t, q_squared):
    return xb * (1. + (t / (2. * q_squared))) / (2. - xb + (xb * t / q_squared))

def compute_t_min(xb, q_squared, ep):
    return -1. * q_squared * ((2. * (1. - xb) * (1. - tf.sqrt(1. + ep**2))) + ep**2) / ((4. * xb * (1. - xb)) + ep**2)

def compute_t_prime(t, tmin):
    return (t-tmin)

def compute_k_tilde(xb, q_squared, t, tmin, ep):
    return tf.sqrt(tmin - t) * tf.sqrt(((1. - xb) * tf.sqrt((1. + ep**2))) + (((tmin - t) * (ep**2 + (4. * (1. - xb) * xb))) / (4. * q_squared)))

def compute_k(q_squared, y_lep, ep, k_tilde):
    return tf.sqrt(((1. - y_lep + (ep**2 * y_lep**2 / 4.)) / q_squared)) * k_tilde

def compute_k_dot_delta(q_squared, xb, t, phi_azi, ep, y_lep, k):
    return (-1.*q_squared / (2.*y_lep*(1.+ep**2))) * (1. + ((2.*k*tf.cos(tf.constant(np.pi) - phi_azi)) - ((t / q_squared)*(1.-(xb * (2. - y_lep)) + (y_lep * ep**2 / 2.))) + (y_lep * ep**2 / 2.)))

def compute_prop_1(q_squared, kdd):
    return (1. + (2. * (kdd / q_squared)))

def compute_prop_2(q_squared, t, kdd):
    return ((-2. * (kdd / q_squared)) + (t / q_squared))

## (5): Training/Testing Data Splitting:

### (5.1): Define the Partition Percentages Here (Train/Validate/Test):

In [ ]:
# https://stackoverflow.com/a/13623707 -> for the typical train/val/test data split
# (we are *not* using the same numbers as recommended here)

TOTAL_DATA_SIZE = NUMBER_OF_PHI_POINTS

_DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE = 0.1 # 90% temporary, 10% testing
_DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE = 0.1 # of the above 90% temporary, 90% training, 10% validation

number_of_dnn_testing_points = np.ceil(TOTAL_DATA_SIZE * _DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE)
number_of_dnn_temporary_points = TOTAL_DATA_SIZE - number_of_dnn_testing_points
number_of_dnn_validation_points = np.ceil(number_of_dnn_temporary_points * _DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE)
number_of_dnn_training_points = number_of_dnn_temporary_points - number_of_dnn_validation_points

print(f"[NOTE]: Testing/Temporary Split is {_DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE * 100}%, giving {number_of_dnn_testing_points} testing points")
print(f"[NOTE]: Training/Validation Split is {_DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE * 100}%, giving {number_of_dnn_validation_points} validation points")
print(f"[NOTE]: Remaining training data points are: {number_of_dnn_training_points}")

### (5.2): Setting up the Input Tensors for TensorFlow:

In [ ]:
k_tf_tensor = np.full(NUMBER_OF_PHI_POINTS, FIXED_K, dtype = np.float32)
t_tf_tensor = np.full(NUMBER_OF_PHI_POINTS, FIXED_T, dtype = np.float32)
xb_tf_tensor = np.full(NUMBER_OF_PHI_POINTS, FIXED_XB, dtype = np.float32)
q2_tf_tensor = np.full(NUMBER_OF_PHI_POINTS, FIXED_Q_SQUARED, dtype = np.float32)

phi_tf_tensor = np.array(phi_array_in_radians, dtype = np.float32)

fe_tf_tensor = compute_fe(t_tf_tensor)
fg_tf_tensor = compute_fg(fe_tf_tensor)
f2_tf_tensor = compute_f2(t_tf_tensor, fe_tf_tensor, fg_tf_tensor)
f1_tf_tensor = compute_f1(fg_tf_tensor, f2_tf_tensor)

epsilon_tf_tensor = compute_epsilon(xb_tf_tensor, q2_tf_tensor)
y_tf_tensor = compute_y(k_tf_tensor, q2_tf_tensor, epsilon_tf_tensor)
xi_tf_tensor = compute_skewness(xb_tf_tensor, t_tf_tensor, q2_tf_tensor)
t_min_tf_tensor = compute_t_min(xb_tf_tensor, q2_tf_tensor, epsilon_tf_tensor)
t_prime_tf_tensor = compute_t_prime(t_tf_tensor, t_min_tf_tensor)
k_tilde_tf_tensor = compute_k_tilde(xb_tf_tensor, q2_tf_tensor, t_tf_tensor, t_min_tf_tensor, epsilon_tf_tensor)
kinematic_k_tf_tensor = compute_k(q2_tf_tensor, y_tf_tensor, epsilon_tf_tensor, k_tilde_tf_tensor)

k_dot_delta_tf_tensor = compute_k_dot_delta(
    q2_tf_tensor, xb_tf_tensor, t_tf_tensor, phi_tf_tensor,
    epsilon_tf_tensor, y_tf_tensor, kinematic_k_tf_tensor)
prop_1_tf_tensor = compute_prop_1(q2_tf_tensor, k_dot_delta_tf_tensor)
prop_2_tf_tensor = compute_prop_2(q2_tf_tensor, t_tf_tensor, k_dot_delta_tf_tensor)

helicity_tf_tensor = np.full(NUMBER_OF_PHI_POINTS, 0.0, dtype = np.float32)
polarization_tf_tensor = np.full(NUMBER_OF_PHI_POINTS, 0.0, dtype = np.float32)

kinematics_and_phi = np.column_stack(
    (
        t_tf_tensor,
        xb_tf_tensor,
        q2_tf_tensor,
        phi_tf_tensor)
    ).astype(np.float32)

physics_data = np.column_stack((
    # kinematics, phi:
    t_tf_tensor,
    xb_tf_tensor,
    q2_tf_tensor,
    phi_tf_tensor,

    # form factors:
    fe_tf_tensor,
    fg_tf_tensor,
    f1_tf_tensor,
    f2_tf_tensor,

    # derived kinematics:
    epsilon_tf_tensor,
    y_tf_tensor,
    xi_tf_tensor,
    t_prime_tf_tensor,
    k_tilde_tf_tensor,
    kinematic_k_tf_tensor,

    # phi-dependent stuff:
    k_dot_delta_tf_tensor,
    prop_1_tf_tensor,
    prop_2_tf_tensor,

    # polarizations:
    helicity_tf_tensor, polarization_tf_tensor
)).astype(np.float32)

In [ ]:
fixed_ep = compute_epsilon(FIXED_XB, FIXED_Q_SQUARED)
fixed_y = compute_y(FIXED_K, FIXED_Q_SQUARED, fixed_ep)
fixed_xi = compute_skewness(FIXED_XB, FIXED_T, FIXED_Q_SQUARED)
fixed_t_min = compute_t_min(FIXED_XB, FIXED_Q_SQUARED, fixed_ep)
fixed_k_tilde = compute_k_tilde(FIXED_XB, FIXED_Q_SQUARED, FIXED_T, fixed_t_min, fixed_ep)
fixed_big_k = compute_k(FIXED_Q_SQUARED, fixed_y, fixed_ep, fixed_k_tilde)
fixed_t_prime = compute_t_prime(FIXED_T, fixed_t_min)
fixed_fe = compute_fe(FIXED_T)
fixed_fg = compute_fg(fixed_fe)
fixed_f2 = compute_f2(FIXED_T, fixed_fe, fixed_fg)
fixed_f1 = compute_f1(fixed_fg, fixed_f2)

print(f"[INFO]: Kinematic Set Number: {100}")
print(f"[INFO]: k = {FIXED_K}")
print(f"[INFO]: Q^2 = {FIXED_Q_SQUARED}")
print(f"[INFO]: xb = {FIXED_XB}")
print(f"[INFO]: t = {FIXED_T}")
print(f"[INFO]: ep: =  {fixed_ep}")
print(f"[INFO]: y = {fixed_y}")
print(f"[INFO]: xi = {fixed_xi}")
print(f"[INFO]: tmin = {fixed_t_min}")
print(f"[INFO]: Ktilde = {fixed_k_tilde}")
print(f"[INFO]: K = {fixed_big_k}")
print(f"[INFO]: tprime = {fixed_t_prime}")
print(f"[INFO]: FE = {fixed_fe}")
print(f"[INFO]: FG = {fixed_fg}")
print(f"[INFO]: F2 = {fixed_f2}")
print(f"[INFO]: F1 = {fixed_f1}")

### (5.3): Here, we tell the DNN what the actual observables to fit are:

In [ ]:
# this is a mega dictionary with all of the bkm10-predicted stuff:
OBSERVABLE_DATA = {
    "unp_beam_unp_target_xsec": bkm10_unp_beam_unp_target_km15,
    "plus_beam_unp_target_xsec": bkm10_plus_beam_unp_target_km15,
    "minus_beam_unp_target_xsec": bkm10_minus_beam_unp_target_km15,
    "unp_beam_plus_target_xsec": bkm10_unp_beam_plus_target_km15,
    "plus_beam_plus_target_xsec": bkm10_plus_beam_plus_target_km15,
    "minus_beam_plus_target_xsec": bkm10_minus_beam_plus_target_km15,
    "unp_beam_minus_target_xsec": bkm10_unp_beam_minus_target_km15,
    "plus_beam_minus_target_xsec": bkm10_plus_beam_minus_target_km15,
    "minus_beam_minus_target_xsec": bkm10_minus_beam_minus_target_km15,
    "unp_target_bsa": bkm10_bsa_unp_target_km15,
    "plus_target_bsa": bkm10_bsa_plus_target_km15,
    "minus_target_bsa": bkm10_bsa_minus_target_km15,
    "unp_beam_tsa": bkm10_unp_beam_tsa_km15,
    "plus_beam_tsa": bkm10_plus_beam_tsa_km15,
    "minus_beam_tsa": bkm10_minus_beam_tsa_km15,
    "dsa": bkm10_dsa_km15,
}

missing_observables = [
    observable for observable in enabled_observables
    if observable not in OBSERVABLE_DATA
]

if missing_observables:
    raise KeyError(
        "[ERROR]: The configuration requested observables that were not "
        f"available in OBSERVABLE_DATA: {missing_observables}"
    )

# stack the observables to fit:
observables_to_learn = np.column_stack((
    [OBSERVABLE_DATA[name] for name in enabled_observables]
)).astype(np.float32)

for index, (name, weight) in enumerate(zip(enabled_observables, OBSERVABLE_WEIGHTS)):
    print(
        f"[INFO]: observable {name} (index = {index}) has fitting weight of"
        f" = {weight:.4f}"
    )

print(f"[INFO]: Array of observables to learn: {observables_to_learn.shape}")

### (5.4): Train/Validation/Testing Splitting here:

In [ ]:
indices_array = np.arange(NUMBER_OF_PHI_POINTS)

dnn_inputs = np.column_stack((t_tf_tensor, xb_tf_tensor, q2_tf_tensor)).astype(np.float32)

training_indices, testing_indices = train_test_split(
    indices_array,
    test_size = 0.10,
    random_state = 7009,)

training_indices, validation_indices = train_test_split(
    training_indices,
    test_size = 0.10,
    random_state = 7009,)

splits = {
    "train": training_indices,
    "validation": validation_indices,
    "test": testing_indices,
}

x_training = dnn_inputs[training_indices]
x_validation = dnn_inputs[validation_indices]
x_testing = dnn_inputs[testing_indices]

precomputed_physics_training = physics_data[training_indices]
precomputed_physics_validation = physics_data[validation_indices]
precomputed_physics_testing = physics_data[testing_indices]

y_training = observables_to_learn[training_indices]
y_validation = observables_to_learn[validation_indices]
y_testing = observables_to_learn[testing_indices]

### (5.5): A collection of effective asserts:

In [ ]:
print(f"[INFO]: Size of x training: {len(x_training)}")
print(f"[INFO]: Does it match with what's expected? {len(x_training) == number_of_dnn_training_points}")

print(f"[INFO]: Size of y training: {len(y_training)}")
print(f"[INFO]: Does it match with what's expected? {len(y_training) == number_of_dnn_training_points}")

print(f"[INFO]: Size of x validation: {len(x_validation)}")
print(f"[INFO]: Does it match with what's expected? {len(x_validation) == number_of_dnn_validation_points}")

print(f"[INFO]: Size of y validation: {len(y_validation)}")
print(f"[INFO]: Does it match with what's expected? {len(y_validation) == number_of_dnn_validation_points}")

print(f"[INFO]: Size of x testing: {len(x_testing)}")
print(f"[INFO]: Does it match with what's expected? {len(x_testing) == number_of_dnn_testing_points}")

print(f"[INFO]: Size of y testing: {len(y_testing)}")
print(f"[INFO]: Does it match with what's expected? {len(y_testing) == number_of_dnn_testing_points}")

### (5.6): Visualizing the Training/Validation/Testing Datapoints with Plots:

#### (5.6.1): A function for plotting the split data:

In [ ]:
def plot_train_validation_test_observable(
    phi,
    observable,
    split_indices,
    ylabel,
    title,
    figure_filename,
):
    
    phi = np.asarray(phi)
    observable = np.asarray(observable)

    figure, axis = plt.subplots(
        nrows = 1,
        ncols = 1,
        figsize = (10, 10)
    )

    number_of_training_points = len(split_indices['train'])
    number_of_validation_points = len(split_indices['validation'])
    number_of_testing_points = len(split_indices['test'])

    print(f"[INFO]: Total training points = {number_of_training_points}")
    print(f"[INFO]: Total validation points = {number_of_validation_points}")
    print(f"[INFO]: Total testing points = {number_of_testing_points}")

    axis.scatter(
        phi[split_indices["train"]],
        observable[split_indices["train"]],
        s = 4.0,
        label = rf"(KM15) Training Points ($N = {number_of_training_points}$)"
    )

    axis.scatter(
        phi[split_indices["validation"]],
        observable[split_indices["validation"]],
        s = 4.0,
        label = rf"(KM15) Validation Points ($N = {number_of_validation_points}$)"
    )

    axis.scatter(
        phi[split_indices["test"]],
        observable[split_indices["test"]],
        s = 4.0,
        label = rf"(KM15) Testing Points ($N = {number_of_testing_points}$)"
    )

    axis.legend(
        fontsize = 20.)

    axis.set_xlabel(
        r"$\phi$ [radians]",
        fontsize = 20.)

    axis.set_ylabel(
        ylabel,
        fontsize = 20.)

    axis.set_title(
        title,
        fontsize = 16.)

    axis.grid(
        visible = True,
        alpha = 0.35)

    for extension in ["png", "eps"]:
        figure.savefig(
            plots_directory /
            f"{figure_filename}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = "white")

    plt.close(figure)

#### (5.6.2): Running the plotting code:

In [ ]:
for observable_name in enabled_observables:

    print(f"[INFO]: Now plotting train/val/test split for observable {observable_name}")

    observable = OBSERVABLE_DATA[observable_name]
    plot_info = observables_to_plot[observable_name]

    plot_train_validation_test_observable(
        phi_array_in_radians,
        observable = observable,
        split_indices = splits,
        ylabel = plot_info["ylabel"],
        title = (
            rf"{observable_name} vs. $\phi$, "
            f"{this_kinematic_set_title_string}"
            "\n"
            f"(KM15): {km15_cff_string}"
        ),
        figure_filename = f"train_test_split_{plot_info['filename']}",
    )

## (6): DNN Model:

### (6.1): Custom DNN Layer that implements the "quenching" procedure:

In [ ]:
@tf.keras.utils.register_keras_serializable(package = "assemble-cffs-layer")
class AssembleCFFs(tf.keras.layers.Layer):

    def __init__(self, free_cff_names, **kwargs):
        super().__init__(**kwargs)

        self.free_cff_names = tuple(free_cff_names)

        self.free_indices = {
            cff_name: index
            for index, cff_name in enumerate(self.free_cff_names)
        }

    def call(self, free_cffs):

        columns = []

        for cff_name in CFF_ORDER:

            if cff_name in self.free_indices:

                free_index = self.free_indices[cff_name]
                column = free_cffs[:, free_index:free_index + 1]

            else:

                column = tf.zeros_like(free_cffs[:, 0:1])

            columns.append(column)

        return tf.concat(columns, axis = 1)

    def get_config(self):

        config = super().get_config()

        config.update({
            "free_cff_names": self.free_cff_names,
        })

        return config

### (6.2): DNN Model Architecture:

In [ ]:
def cff_dnn_model():

    kinematics_inputs = tf.keras.Input(
        shape = (3,),
        name = "input_values")
    
    physics_input = tf.keras.Input(
        shape = (19,),
        name = "precomputed_physics")
    
    dnn_kinematic_inputs = tf.keras.layers.Lambda(
        lambda x: x[:, :3],
        name = "input_kinematics"
    )(kinematics_inputs)
    
    hidden = tf.keras.layers.Dense(
        _NUMBER_OF_NODES_LAYER_1,
        kernel_initializer = "he_normal",
        activation = "relu"
    )(dnn_kinematic_inputs)
    
    hidden = tf.keras.layers.Dense(
        _NUMBER_OF_NODES_LAYER_2,
        kernel_initializer = "he_normal",
        activation = "relu"
    )(hidden)
    
    hidden = tf.keras.layers.Dense(
        _NUMBER_OF_NODES_LAYER_3,
        kernel_initializer = "he_normal",
        activation = "relu"
    )(hidden)
    
    hidden = tf.keras.layers.Dense(
        _NUMBER_OF_NODES_LAYER_4,
        kernel_initializer = "he_normal",
        activation = "relu"
    )(hidden)
    
    # the output dimensionalty is dependent on this variable we
    # computed at the beginning...
    cff_free_outputs = tf.keras.layers.Dense(
        NUMBER_OF_FREE_CFFS,
        activation = "linear",
        name = "cff_free_outputs"
    )(hidden)

    cff_outputs = AssembleCFFs(
        FREE_CFF_NAMES,
        name = "cff_outputs"
    )(cff_free_outputs)

    full_model_outputs = tf.keras.layers.Concatenate(
        name = "physics_and_cffs")([
        cff_outputs,
        physics_input
    ])
    
    model = tf.keras.Model(
        inputs = [
            kinematics_inputs,
            physics_input
            ],
        outputs = full_model_outputs)

    model.compile(
        optimizer = tf.keras.optimizers.Adam(
            learning_rate = LEARNING_RATE
            ),
        loss = UnfoldedSimultaneousFitLoss(
            enabled_observables = enabled_observables,
            observable_weights = OBSERVABLE_WEIGHTS
        ),
        jit_compile = True,
        )

    return model

## (7): Fitting the Model:

### (7.1): Get a TF Summary of the DNN Model:

In [ ]:
cff_dnn_model().summary()

### (7.2): **Begin the Replica Method**

In [ ]:
for replica in range(NUMBER_OF_REPLICAS):
    
    replica_number = replica + 1

    start_dnn_compile_time = time.perf_counter()
    dnn_model = cff_dnn_model()
    print(f"[TIMING]: model construction time: {time.perf_counter() - start_dnn_compile_time:.2f} s")

    start_dnn_fitting_time = time.perf_counter()
    dnn_model_history = dnn_model.fit(
        x = {
            "input_values": x_training,
            "precomputed_physics": precomputed_physics_training
        },
        y = y_training,
        validation_data = (
            {
                "input_values": x_validation,
                "precomputed_physics": precomputed_physics_validation
            },
            y_validation
        ),
        epochs = NUMBER_OF_EPOCHS,
        batch_size = BATCH_SIZE,
        verbose = 0
    )
    print(f"[TIMING]: fitting time: {time.perf_counter() - start_dnn_fitting_time:.2f} s")

    number_of_epochs_run = len(dnn_model_history.epoch)
    print(f"[INFO]: The model ran for {number_of_epochs_run} epochs.")

    start_dnn_save_time = time.perf_counter()
    dnn_model.save(
        replica_directory /
        f"replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.keras"
        )
    print(f"[TIMING]: DNN saving time: {time.perf_counter() - start_dnn_save_time:.2f} s")

    training_loss_data = dnn_model_history.history["loss"]
    validation_loss_data = dnn_model_history.history["val_loss"]

    start_dnn_evaluation_time = time.perf_counter()
    dnn_evaluation_statistics = dnn_model.evaluate(
        x = {
            "input_values": x_testing,
            "precomputed_physics": precomputed_physics_testing
            },
        y = y_testing,
        verbose = 0)
    print(f"[TIMING]: DNN evaluation time: {time.perf_counter() - start_dnn_evaluation_time:.2f} s")
    print(f"[INFO]: Test Loss for Replica {replica_number}: {dnn_evaluation_statistics}")

    # make DF with testing metrics:
    testing_loss_information = pd.DataFrame({
        # https://stackoverflow.com/a/17840195 -> for why we need to cast it into a list!
        'testing_loss': [dnn_evaluation_statistics],
    })

    testing_loss_information.to_csv(
        replica_directory /
        f"replica_{replica_number}_loss_data_v{MAJOR_MINOR_NUMBER}.csv",
        index = False
    )

    # save npz with DNN training information:
    np.savez(
        file = (
            replica_directory /
            f"replica_{replica_number}_losses_vs_epochs_v{MAJOR_MINOR_NUMBER}.npz"
            ),
        training_loss = training_loss_data,
        validation_loss = validation_loss_data
        )

    # make DF with DNN training information
    pd.DataFrame(dnn_model_history.history).to_csv(
        replica_directory /
        f"replica_{replica_number}_losses_vs_epochs_v{MAJOR_MINOR_NUMBER}.csv",
        index = False)

    predicted_outputs = dnn_model.predict(
        {
            "input_values": dnn_inputs,
            "precomputed_physics": physics_data,
        },
        verbose = 0
    )

    cff_predictions = predicted_outputs[:, :8]

    cff_h_real = cff_predictions[:, 0]
    cff_h_imag = cff_predictions[:, 1]
    cff_ht_real = cff_predictions[:, 2]
    cff_ht_imag = cff_predictions[:, 3]
    cff_e_real = cff_predictions[:, 4]
    cff_e_imag = cff_predictions[:, 5]
    cff_et_real = cff_predictions[:, 6]
    cff_et_imag = cff_predictions[:, 7]

    fe = compute_fe(t_tf_tensor)
    fg = compute_fg(fe)
    f2 = compute_f2(t_tf_tensor, fe, fg)
    f1 = compute_f1(fg, f2)
    epsilon = compute_epsilon(xb_tf_tensor, q2_tf_tensor)
    y_lep = compute_y(FIXED_K, q2_tf_tensor, epsilon)
    xi = compute_skewness(xb_tf_tensor, t_tf_tensor, q2_tf_tensor)
    tmin = compute_t_min(xb_tf_tensor, q2_tf_tensor, epsilon)
    tprime = compute_t_prime(t_tf_tensor, tmin)
    k_tilde = compute_k_tilde(xb_tf_tensor, q2_tf_tensor, t_tf_tensor, tmin, epsilon)
    kinematic_k = compute_k(q2_tf_tensor, y_lep, epsilon, k_tilde)
    kdd = compute_k_dot_delta(q2_tf_tensor, xb_tf_tensor, t_tf_tensor, phi_tf_tensor, epsilon, y_lep, kinematic_k)
    p1 = compute_prop_1(q2_tf_tensor, kdd)
    p2 = compute_prop_2(q2_tf_tensor, t_tf_tensor, kdd)

    sigma_plus_plus = bkm10_cross_section(
        +1.0, +0.5,
        q2_tf_tensor, xb_tf_tensor, t_tf_tensor, epsilon, y_lep, xi, kinematic_k, f1, f2, k_tilde, tprime, phi_tf_tensor, p1, p2,
        cff_h_real, cff_ht_real, cff_e_real, cff_et_real, cff_h_imag, cff_ht_imag, cff_e_imag, cff_et_imag)
    sigma_minus_plus = bkm10_cross_section(
        -1.0, +0.5,
        q2_tf_tensor, xb_tf_tensor, t_tf_tensor, epsilon, y_lep, xi, kinematic_k, f1, f2, k_tilde, tprime, phi_tf_tensor, p1, p2,
        cff_h_real, cff_ht_real, cff_e_real, cff_et_real, cff_h_imag, cff_ht_imag, cff_e_imag, cff_et_imag)
    sigma_plus_minus = bkm10_cross_section(
        +1.0, -0.5,
        q2_tf_tensor, xb_tf_tensor, t_tf_tensor, epsilon, y_lep, xi, kinematic_k, f1, f2, k_tilde, tprime, phi_tf_tensor, p1, p2,
        cff_h_real, cff_ht_real, cff_e_real, cff_et_real, cff_h_imag, cff_ht_imag, cff_e_imag, cff_et_imag)
    sigma_minus_minus = bkm10_cross_section(
        -1.0, -0.5,
        q2_tf_tensor, xb_tf_tensor, t_tf_tensor, epsilon, y_lep, xi, kinematic_k, f1, f2, k_tilde, tprime, phi_tf_tensor, p1, p2,
        cff_h_real, cff_ht_real, cff_e_real, cff_et_real, cff_h_imag, cff_ht_imag, cff_e_imag, cff_et_imag)

    # cross-section | sigma(0, 0):
    sigma_unp_beam_unp_target = (0.25 * (
        sigma_plus_plus + sigma_plus_minus + sigma_minus_plus + sigma_minus_minus
    ))

    # cross-section | sigma(+1, 0):
    sigma_plus_beam_unp_target = (0.5 * (sigma_plus_plus + sigma_plus_minus))
    # cross-section | sigma(-1, 0):
    sigma_minus_beam_unp_target = (0.5 * (sigma_minus_plus + sigma_minus_minus))

    # cross-section | sigma(0, +1/2):
    sigma_unp_beam_plus_target = (0.5 * (sigma_plus_plus + sigma_minus_plus))
    # cross-section | sigma(0, -1/2):
    sigma_unp_beam_minus_target = (0.5 * (sigma_plus_minus + sigma_minus_minus))

    # cross-section | sigma(+1, +1/2):
    sigma_plus_beam_plus_target = sigma_plus_plus
    # cross-section | sigma(-1, +1/2):
    sigma_minus_beam_plus_target = sigma_minus_plus
    # cross-section | sigma(+1, -1/2):
    sigma_plus_beam_minus_target = sigma_plus_minus
    # cross-section | sigma(-1, +1/2):
    sigma_minus_beam_minus_target = sigma_minus_minus

    # sum of all sigma contributions:
    total_sigma = (
        sigma_plus_plus + sigma_plus_minus + sigma_minus_plus + sigma_minus_minus
    )

    # BSA(0)
    bsa_unp_target = ((
        sigma_plus_plus + sigma_plus_minus - sigma_minus_plus - sigma_minus_minus
    ) / total_sigma)

    # BSA(+1/2)
    bsa_plus_target = (sigma_plus_plus - sigma_minus_plus) / (sigma_plus_plus + sigma_minus_plus)

    # BSA(-1/2)
    bsa_minus_target = (sigma_plus_minus - sigma_minus_minus) / (sigma_plus_minus + sigma_minus_minus)

    # TSA(0)
    tsa_unp_beam = (
        (sigma_plus_plus + sigma_minus_plus - sigma_plus_minus - sigma_minus_minus) / 
        total_sigma)

    # TSA(+1)
    tsa_plus_beam = (
        (sigma_plus_plus - sigma_plus_minus) /
        (sigma_plus_plus + sigma_plus_minus))

    # TSA(-1):
    tsa_minus_beam = (
        (sigma_minus_plus - sigma_minus_minus) /
        (sigma_minus_plus + sigma_minus_minus))

    # DSA
    dsa = ((
        sigma_plus_plus - sigma_plus_minus- sigma_minus_plus + sigma_minus_minus
    ) / total_sigma)

    np.savez(
        # filename:
        file =
            (
                replica_directory /
                f"replica_{replica_number}_predictions_v{MAJOR_MINOR_NUMBER}.npz"
            ),

        # kinematics:
        t = t_tf_tensor,
        xb = xb_tf_tensor,
        q_squared = q2_tf_tensor,
        phi = phi_tf_tensor,

        # save CFF data:
        cff_h_real = cff_h_real,
        cff_h_imag = cff_h_imag,
        cff_ht_real = cff_ht_real,
        cff_ht_imag = cff_ht_imag,
        cff_e_real = cff_e_real,
        cff_e_imag = cff_e_imag,
        cff_et_real = cff_et_real,
        cff_et_imag = cff_et_imag,

        sigma_plus_plus = sigma_plus_plus,
        sigma_minus_plus = sigma_minus_plus,
        sigma_plus_minus = sigma_plus_minus,
        sigma_minus_minus = sigma_minus_minus,

        unp_beam_unp_target_xsec = sigma_unp_beam_unp_target,

        plus_beam_unp_target_xsec = sigma_plus_beam_unp_target,
        minus_beam_unp_target_xsec = sigma_minus_beam_unp_target,

        unp_beam_plus_target_xsec = sigma_unp_beam_plus_target,
        plus_beam_plus_target_xsec = sigma_plus_beam_plus_target,
        minus_beam_plus_target_xsec = sigma_minus_beam_plus_target,

        unp_beam_minus_target_xsec = sigma_unp_beam_minus_target,
        plus_beam_minus_target_xsec = sigma_plus_beam_minus_target,
        minus_beam_minus_target_xsec = sigma_minus_beam_minus_target,

        unp_target_bsa = bsa_unp_target,
        plus_target_bsa = bsa_plus_target,
        minus_target_bsa = bsa_minus_target,

        unp_beam_tsa = tsa_unp_beam,
        plus_beam_tsa = tsa_plus_beam,
        minus_beam_tsa = tsa_minus_beam,

        dsa = dsa,

        enabled_observables = np.asarray(
            enabled_observables,
            dtype = "U"
        ),

        observable_weights = np.asarray(
            OBSERVABLE_WEIGHTS,
            dtype = np.float64
        ))

    # [INFO]: this is called an "aggressive" memory cleanup:
    del dnn_model
    del dnn_model_history
    del training_loss_data
    del validation_loss_data
    del dnn_evaluation_statistics

    gc.collect()
    tf.keras.backend.clear_session()

### (7.3): Make Loss Plots for Every Replica:

In [ ]:
# just do this for now hahahaha:
_REGULARIZER = 1e-21

for loss_file in sorted(
    replica_directory.glob(F"replica_*_losses_vs_epochs_v{MAJOR_MINOR_NUMBER}.csv")
):

    replica_match = re.search(r"replica_(\d+)_losses_vs_epochs\.csv", loss_file.name)

    if replica_match is None:
        continue

    replica_number = int(replica_match.group(1))

    loss_information = pd.read_csv(loss_file)
    
    training_loss_data = loss_information["loss"].to_numpy()
    validation_loss_data = loss_information["val_loss"].to_numpy()

    number_of_epochs_run = len(loss_information)
    epoch_array = np.arange(number_of_epochs_run)

    testing_loss_information = pd.read_csv(
        replica_directory /
        f"replica_{replica_number}_loss_data.csv")

    testing_loss = testing_loss_information["testing_loss"].iloc[0]

    initial_loss = np.max(training_loss_data)

    curves_fig, curves_ax = plt.subplots(
        nrows = 1,
        ncols = 1,
        figsize = (10, 10))
    
    log_curves_fig, log_curves_ax = plt.subplots(
        nrows = 1,
        ncols = 1,
        figsize = (10, 10))

    curves_ax.plot(
        epoch_array,
        np.full(number_of_epochs_run, initial_loss),
        color = "red",
        label = "Initial Loss Value")
    
    curves_ax.plot(
        epoch_array,
        np.zeros(number_of_epochs_run),
        color = "green",
        label = r"Loss $= 0$")
    
    curves_ax.plot(
        epoch_array,
        training_loss_data,
        color = "blue",
        label = "Training Loss")
    
    curves_ax.plot(
        epoch_array,
        validation_loss_data,
        color = "purple",
        label = "Validation Loss")

    curves_ax.legend(
        fontsize = 16.)

    curves_ax.set_xlabel(
        "Epoch",
        fontsize = 20.)
    
    curves_ax.set_ylabel(
        "MSE",
        fontsize = 20.)
    
    curves_ax.set_title(
        f"Replica {replica_number} Learning Curves\n(Test Loss $= {testing_loss:.3g}$)",
        fontsize = 20.)
    
    log_curves_ax.plot(
        epoch_array,
        np.log(np.full(number_of_epochs_run, initial_loss)),
        color = "red",
        label = "Initial Loss Value")
    
    log_curves_ax.plot(
        epoch_array,
        np.log(np.full(number_of_epochs_run, _REGULARIZER)),
        color = "green",
        label = r"Loss $= 0$")
    
    log_curves_ax.plot(
        epoch_array,
        np.log(training_loss_data + _REGULARIZER),
        color = "blue",
        label = "Log Training Loss")
    
    log_curves_ax.plot(
        epoch_array,
        np.log(validation_loss_data + _REGULARIZER),
        color = "purple",
        label = "Log Validation Loss")

    log_curves_ax.legend(
        fontsize = 16.)

    log_curves_ax.set_xlabel(
        "Epoch", 
        fontsize = 20.)
    
    log_curves_ax.set_ylabel(
        "Log MSE Loss", 
        fontsize = 20.)
    
    log_curves_ax.set_title(
        f"Replica {replica_number} Learning Curves\n(Test Loss $= {testing_loss:.3g}$)",
        fontsize = 20.)

    for extension in ['png', 'eps']:
        curves_fig.savefig(
            learning_curves_directory /
            f"lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white')

        log_curves_fig.savefig(
            learning_curves_directory /
            f"log_lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white')

    plt.close(curves_fig)
    plt.close(log_curves_fig)

    del curves_fig
    del log_curves_fig

## (8): DNN Predictions:

### (8.1): Prepare Smooth Input Data for DNN Predictions:

#### (8.1.1): Smooth $x_{\text{B}}$, $Q^{2}$, and $t$ values (for smooth grid representation):

In [ ]:
range_of_t = np.linspace(x_training.min(), x_training.max())
range_of_x_b = np.linspace(x_training.min(), x_training.max())
range_of_q_squared = np.linspace(x_training.min(), x_training.max())

### (8.2): Extract all of the Keras replica files:

In [ ]:
replica_paths = sorted(
    replica_directory.glob(f"replica_*_v{MAJOR_MINOR_NUMBER}.keras")
)

replicas = [tf.keras.models.load_model(
    path,
    compile = False,
    safe_mode = False) for path in replica_paths]

print(f"[INFO]: Loaded {len(replicas)} replica models.")

assert len(replicas) == NUMBER_OF_REPLICAS, "[ASSERT]: Number of loaded replicas does not equal expected number"

### (8.3): Firstly, we simply *extract* what the replicas predicted for each thing, including CFFs and observables

In [ ]:
# cross-section | sigma(0, 0)
replicas_unp_beam_unp_target_xsec = []

# cross-section | sigma(+1, 0)
replicas_plus_beam_unp_target_xsec = []

# cross-section | sigma(-1, 0)
replicas_minus_beam_unp_target_xsec = []

# cross-section | sigma(0, +1/2)
replicas_unp_beam_plus_target_xsec = []

# cross-section | sigma(+1, +1/2)
replicas_plus_beam_plus_target_xsec = []

# cross-section | sigma(-1, +1/2)
replicas_minus_beam_plus_target_xsec = []

# cross-section | sigma(0, -1/2)
replicas_unp_beam_minus_target_xsec = []

# cross-section | sigma(+1, -1/2)
replicas_plus_beam_minus_target_xsec = []

# cross-section | sigma(-1, -1/2)
replicas_minus_beam_minus_target_xsec = []

# BSA(0)
replicas_unp_target_bsa = []

# BSA(+1/2)
replicas_plus_target_bsa = []

# BSA(-1/2)
replicas_minus_target_bsa = []

# TSA(0)
replicas_unp_beam_tsa = []

# TSA(+1)
replicas_plus_beam_tsa = []

# TSA(-1)
replicas_minus_beam_tsa = []

# DSA
replicas_dsa = []

# CFF stuff:
replicas_h_real = []
replicas_h_imag = []
replicas_ht_real = []
replicas_ht_imag = []
replicas_e_real = []
replicas_e_imag = []
replicas_et_real = []
replicas_et_imag = []

for replica_index in range(1, NUMBER_OF_REPLICAS + 1):

    prediction_path = (
        replica_directory
        / f"replica_{replica_index}_predictions_v{MAJOR_MINOR_NUMBER}.npz"
    )

    # [NOTE]: recall the npz file is a dictionary of np arrays, so
    # you need to remember their keys:
    prediction_information = np.load(prediction_path)

    # reading CFF prediction data:
    replicas_h_real.append(prediction_information["cff_h_real"])
    replicas_h_imag.append(prediction_information["cff_h_imag"])
    replicas_ht_real.append(prediction_information["cff_ht_real"])
    replicas_ht_imag.append(prediction_information["cff_ht_imag"])
    replicas_e_real.append(prediction_information["cff_e_real"])
    replicas_e_imag.append(prediction_information["cff_e_imag"])
    replicas_et_real.append(prediction_information["cff_et_real"])
    replicas_et_imag.append(prediction_information["cff_et_imag"])

    replicas_unp_beam_unp_target_xsec.append(
        prediction_information["unp_beam_unp_target_xsec"]
    )

    replicas_plus_beam_unp_target_xsec.append(
        prediction_information["plus_beam_unp_target_xsec"]
    )

    replicas_minus_beam_unp_target_xsec.append(
        prediction_information["minus_beam_unp_target_xsec"]
    )

    replicas_unp_beam_plus_target_xsec.append(
        prediction_information["unp_beam_plus_target_xsec"]
    )

    replicas_plus_beam_plus_target_xsec.append(
        prediction_information["plus_beam_plus_target_xsec"]
    )

    replicas_minus_beam_plus_target_xsec.append(
        prediction_information["minus_beam_plus_target_xsec"]
    )

    replicas_unp_beam_minus_target_xsec.append(
        prediction_information["unp_beam_minus_target_xsec"]
    )

    replicas_plus_beam_minus_target_xsec.append(
        prediction_information["plus_beam_minus_target_xsec"]
    )

    replicas_minus_beam_minus_target_xsec.append(
        prediction_information["minus_beam_minus_target_xsec"]
    )

    replicas_unp_target_bsa.append(
        prediction_information["unp_target_bsa"]
    )

    replicas_plus_target_bsa.append(
        prediction_information["plus_target_bsa"]
    )

    replicas_minus_target_bsa.append(
        prediction_information["minus_target_bsa"]
    )

    replicas_unp_beam_tsa.append(
        prediction_information["unp_beam_tsa"]
    )

    replicas_plus_beam_tsa.append(
        prediction_information["plus_beam_tsa"]
    )

    replicas_minus_beam_tsa.append(
        prediction_information["minus_beam_tsa"]
    )

    replicas_dsa.append(
        prediction_information["dsa"]
    )

    prediction_information.close()

### (8.4): Secondly, we need to evaluate the statistics for this replica distribution:

#### (8.4.1): A function that computes the statistics across every replica:

In [ ]:
def crunch_statistics(data):
    
    # initialize dictionary
    statistics_dictionary = {}

    # statistical parameter:
    statistics_dictionary["mean"] = np.mean(data, axis = 0)
    statistics_dictionary["std"] = np.std(data, axis = 0)
    statistics_dictionary["median"] = np.median(data, axis = 0)
    statistics_dictionary["max"] = np.max(data, axis = 0)
    statistics_dictionary["min"] = np.min(data, axis = 0)
    
    # percentiles:
    statistics_dictionary['p10'] = np.percentile(data, 10, axis = 0)
    statistics_dictionary['p20'] = np.percentile(data, 20, axis = 0)
    statistics_dictionary['p30'] = np.percentile(data, 30, axis = 0)
    statistics_dictionary['p40'] = np.percentile(data, 40, axis = 0)
    statistics_dictionary['p50'] = np.percentile(data, 50, axis = 0)
    statistics_dictionary['p60'] = np.percentile(data, 60, axis = 0)
    statistics_dictionary['p70'] = np.percentile(data, 70, axis = 0)
    statistics_dictionary['p80'] = np.percentile(data, 80, axis = 0)
    statistics_dictionary['p90'] = np.percentile(data, 90, axis = 0)

    return statistics_dictionary

#### (8.4.2): A huge dictionary that contains ever replica statistical distribution per observable:

In [ ]:
# a huge dictionary of all of the replica statistical distributions 
# per observable:
observable_replica_predictions = {
    "unp_beam_unp_target_xsec": replicas_unp_beam_unp_target_xsec,
    "plus_beam_unp_target_xsec": replicas_plus_beam_unp_target_xsec,
    "minus_beam_unp_target_xsec": replicas_minus_beam_unp_target_xsec,

    "unp_beam_plus_target_xsec": replicas_unp_beam_plus_target_xsec,
    "plus_beam_plus_target_xsec": replicas_plus_beam_plus_target_xsec,
    "minus_beam_plus_target_xsec": replicas_minus_beam_plus_target_xsec,

    "unp_beam_minus_target_xsec": replicas_unp_beam_minus_target_xsec,
    "plus_beam_minus_target_xsec": replicas_plus_beam_minus_target_xsec,
    "minus_beam_minus_target_xsec": replicas_minus_beam_minus_target_xsec,

    "unp_target_bsa": replicas_unp_target_bsa,
    "plus_target_bsa": replicas_plus_target_bsa,
    "minus_target_bsa": replicas_minus_target_bsa,

    "unp_beam_tsa": replicas_unp_beam_tsa,
    "plus_beam_tsa": replicas_plus_beam_tsa,
    "minus_beam_tsa": replicas_minus_beam_tsa,

    "dsa": replicas_dsa,
}

#### (8.4.3): Computing the Observables Statistical Distributions:

In [ ]:
# the idea here is to compute the statistics for
# all of the predicted observables...
observable_statistics = {
    observable_name: crunch_statistics(predictions)
    for observable_name, predictions in observable_replica_predictions.items()
}

assert set(OBSERVABLE_DATA) == set(observable_replica_predictions)

missing_from_truth = (
    set(observable_replica_predictions) - set(OBSERVABLE_DATA)
)

missing_from_predictions = (
    set(OBSERVABLE_DATA) - set(observable_replica_predictions)
)

if missing_from_truth:
    raise KeyError(
        "[ERROR]: These predicted observables have no corresponding "
        f"BKM10 truth data: {sorted(missing_from_truth)}"
    )

if missing_from_predictions:
    raise KeyError(
        "[ERROR]: These BKM10 observables have no corresponding "
        f"replica predictions: {sorted(missing_from_predictions)}"
    )

# initialize the dataframe:
replica_statistics_dataframe = pd.DataFrame({
    'k': FIXED_K,
    't': FIXED_T,
    'xb': FIXED_XB,
    'q_squared': FIXED_Q_SQUARED,
    'phi': phi_array_in_degrees,
    "Re[H]": CFF_REAL_H_KM15,
    "Im[H]": CFF_IMAG_H_KM15,
    "Re[E]": CFF_REAL_E_KM15,
    "Im[E]": CFF_IMAG_E_KM15,
    "Re[Ht]": CFF_REAL_HT_KM15,
    "Im[Ht]": CFF_IMAG_HT_KM15,
    "Re[Et]": CFF_REAL_ET_KM15, 
    "Im[Et]": CFF_IMAG_ET_KM15,

})

# now add to the dataframe all of the predictions:
for observable_name, statistics in observable_statistics.items():

    for statistic_name, values in statistics.items():

        # adding an associated key to the DF:
        replica_statistics_dataframe[f"{statistic_name}_{observable_name}"] = values

# save the DF:
replica_statistics_dataframe.to_csv(
    path_or_buf = (
        data_directory /
        f"observable_replica_predictions_v{MAJOR_MINOR_NUMBER}.csv"
        ),
    index = False)

#### (8.4.4): Computing the CFF Statistical Distributions:

In [ ]:
cff_h_real_pred_per_replica = np.mean(replicas_h_real, axis = 1)
cff_h_imag_pred_per_replica = np.mean(replicas_h_imag, axis = 1)
cff_ht_real_pred_per_replica = np.mean(replicas_ht_real, axis = 1)
cff_ht_imag_pred_per_replica = np.mean(replicas_ht_imag, axis = 1)
cff_e_real_pred_per_replica = np.mean(replicas_e_real, axis = 1)
cff_e_imag_pred_per_replica = np.mean(replicas_e_imag, axis = 1)
cff_et_real_pred_per_replica = np.mean(replicas_et_real, axis = 1)
cff_et_imag_pred_per_replica = np.mean(replicas_et_imag, axis = 1)

replica_cffs_dataframe = pd.DataFrame({
    "ReH_pred": cff_h_real_pred_per_replica,
    "ImH_pred": cff_h_imag_pred_per_replica,

    "ReHt_pred": cff_ht_real_pred_per_replica,
    "ImHt_pred": cff_ht_imag_pred_per_replica,

    "ReE_pred": cff_e_real_pred_per_replica,
    "ImE_pred": cff_e_imag_pred_per_replica,

    "ReEt_pred": cff_et_real_pred_per_replica,
    "ImEt_pred": cff_et_imag_pred_per_replica,
})

replica_cffs_dataframe.to_csv(
    data_directory /
    f"cff_replica_predictions_v{MAJOR_MINOR_NUMBER}.csv", 
    index = False)

## (9): Plots of the DNN Predictions:

### (9.1): Huge dictionary of all the observables to plot:

In [ ]:
OBSERVABLE_PLOT_INFO = {
    "unp_beam_unp_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{UU}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{UU}$ vs. $\phi$",
        "filename": "dnn_unp_beam_unp_target_xsec_vs_phi",
    },

    "plus_beam_unp_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{+U}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{+U}$ vs. $\phi$",
        "filename": "dnn_plus_beam_unp_target_xsec_vs_phi",
    },

    "minus_beam_unp_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{-U}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{-U}$ vs. $\phi$",
        "filename": "dnn_minus_beam_unp_target_xsec_vs_phi",
    },

    "unp_beam_plus_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{UL}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{UL}$ vs. $\phi$",
        "filename": "dnn_unp_beam_plus_target_xsec_vs_phi",
    },

    "plus_beam_plus_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{+L}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{+L}$ vs. $\phi$",
        "filename": "dnn_plus_beam_plus_target_xsec_vs_phi",
    },

    "minus_beam_plus_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{-L}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{-L}$ vs. $\phi$",
        "filename": "dnn_minus_beam_plus_target_xsec_vs_phi",
    },

    "unp_beam_minus_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{UL}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{UL}$ vs. $\phi$",
        "filename": "dnn_unp_beam_minus_target_xsec_vs_phi",
    },

    "plus_beam_minus_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{+L}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{+L}$ vs. $\phi$",
        "filename": "dnn_plus_beam_minus_target_xsec_vs_phi",
    },

    "minus_beam_minus_target_xsec": {
        "ylabel": r"$d^{4}\sigma^{-L}$ [nb / GeV$^{4}$]",
        "title": r"$d^{4}\sigma^{-L}$ vs. $\phi$",
        "filename": "dnn_minus_beam_minus_target_xsec_vs_phi",
    },

    "unp_target_bsa": {
        "ylabel": r"$A_{LU}$",
        "title": r"Unpolarized-target BSA vs. $\phi$",
        "filename": "dnn_unp_target_bsa_vs_phi",
    },

    "plus_target_bsa": {
        "ylabel": r"$A_{LU}^{+}$",
        "title": r"Positive-target-polarization BSA vs. $\phi$",
        "filename": "dnn_plus_lp_target_bsa_vs_phi",
    },

    "minus_target_bsa": {
        "ylabel": r"$A_{LU}^{-}$",
        "title": r"Negative-target-polarization BSA vs. $\phi$",
        "filename": "dnn_minus_lp_target_bsa_vs_phi",
    },

    "unp_beam_tsa": {
        "ylabel": r"$A_{UT}$",
        "title": r"Unpolarized-beam TSA vs. $\phi$",
        "filename": "dnn_unp_beam_tsa_vs_phi",
    },

    "plus_beam_tsa": {
        "ylabel": r"$A_{UT}^{+}$",
        "title": r"Positive-beam TSA vs. $\phi$",
        "filename": "dnn_plus_beam_tsa_vs_phi",
    },

    "minus_beam_tsa": {
        "ylabel": r"$A_{UT}^{-}$",
        "title": r"Negative-beam TSA vs. $\phi$",
        "filename": "dnn_minus_beam_tsa_vs_phi",
    },

    "dsa": {
        "ylabel": r"$A_{LL}$",
        "title": r"Double-spin asymmetry vs. $\phi$",
        "filename": "dnn_dsa_vs_phi",
    },
}

### (9.2): A function that shows how the DNN performed vs. the true data:

In [ ]:
def plot_observable_comparison(
    observable_name,
    phi_array,
    observable_data,
    statistics_dataframe,
    this_kinematic_set_title_string,
    km15_cff_string,
):
    """
    Plot the BKM10/KM15 observable against the replica-average DNN
    prediction and its statistical bounds.
    """

    plot_info = OBSERVABLE_PLOT_INFO[observable_name]

    mean_column = f"mean_{observable_name}"
    max_column = f"max_{observable_name}"
    min_column = f"min_{observable_name}"
    p90_column = f"p90_{observable_name}"
    p10_column = f"p10_{observable_name}"
    p80_column = f"p80_{observable_name}"
    p20_column = f"p20_{observable_name}"
    p70_column = f"p70_{observable_name}"
    p30_column = f"p30_{observable_name}"
    p60_column = f"p60_{observable_name}"
    p40_column = f"p40_{observable_name}"

    required_columns = [
        mean_column,
        max_column,
        min_column,
        p90_column,
        p10_column,
        p80_column,
        p20_column,
        p70_column,
        p30_column,
        p60_column,
        p40_column,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in statistics_dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"[ERROR]: Missing statistics columns for "
            f"'{observable_name}': {missing_columns}"
        )

    if observable_name not in observable_data:
        raise KeyError(
            f"[ERROR]: No BKM10 prediction available for "
            f"'{observable_name}'."
        )

    fig, ax = plt.subplots(
        nrows = 1,
        ncols = 1,
        figsize = (11, 10))

    ax.scatter(
        phi_array,
        observable_data[observable_name],
        s = 4.,
        label = "BKM10 Prediction with KM15 CFFs",
    )

    ax.plot(
        phi_array,
        statistics_dataframe[mean_column],
        label = r"Replica Average",
        linewidth = 0.5,
        linestyle = "dashed",)

    ax.fill_between(
        x = phi_array,
        y1 = statistics_dataframe[max_column],
        y2 = statistics_dataframe[min_column],
        label = r"Min/Max Bound",
        alpha = 0.2,)

    ax.fill_between(
        x = phi_array,
        y1 = statistics_dataframe[p90_column],
        y2 = statistics_dataframe[p10_column],
        label = r"10/90 \% Bound",
        alpha = 0.25,)

    ax.fill_between(
        x = phi_array,
        y1 = statistics_dataframe[p80_column],
        y2 = statistics_dataframe[p20_column],
        label = r"20/80 \% Bound",
        alpha = 0.3,)

    ax.fill_between(
        x = phi_array,
        y1 = statistics_dataframe[p70_column],
        y2 = statistics_dataframe[p30_column],
        label = r"30/70 \% Bound",
        alpha = 0.35,)

    ax.fill_between(
        x = phi_array,
        y1 = statistics_dataframe[p60_column],
        y2 = statistics_dataframe[p40_column],
        label = r"40/60 \% Bound",
        alpha = 0.4,)

    ax.set_xlabel(
        r"$\phi$ [radians]",
        fontsize = 20.)

    ax.set_ylabel(
        plot_info["ylabel"],
        fontsize = 20.)

    ax.set_title(
        f'{plot_info["title"]}, {this_kinematic_set_title_string}'
        "\n"
        f"(KM15): {km15_cff_string}",
        fontsize = 18.
    )

    ax.legend(fontsize = 20.)

    fig.tight_layout()

    for extension in ["png", "eps"]:
        fig.savefig(
            plots_directory /
            f'{plot_info["filename"]}_v{MAJOR_MINOR_NUMBER}.{extension}',
            facecolor = "white"
        )

    plt.close(fig)

### (9.3): A plot explicitly showing the residuals of the DNN fitting procedure:

In [ ]:
def plot_observable_residuals(
    observable_name,
    phi_array,
    observable_data,
    statistics_dataframe,
    this_kinematic_set_title_string,
    km15_cff_string,):
    """
    Plot the replica-average residual for a single observable.
    """

    plot_info = OBSERVABLE_PLOT_INFO[observable_name]

    mean_column = f"mean_{observable_name}"

    if mean_column not in statistics_dataframe.columns:
        raise KeyError(
            f"[ERROR]: Missing mean statistics column: {mean_column}"
        )

    residual = (
        statistics_dataframe[mean_column].to_numpy()
        - observable_data[observable_name]
    )

    fig, ax = plt.subplots(
        nrows = 2,
        ncols = 1,
        gridspec_kw = {"height_ratios": [3, 1]},
        figsize = (11, 10))

    ax[0].scatter(
        phi_array,
        observable_data[observable_name],
        s = 4.,
        label = "BKM10 Prediction with KM15 CFFs",)

    ax[0].plot(
        phi_array,
        statistics_dataframe[mean_column],
        label = "Replica Average",
        linewidth = 0.5,
        linestyle = "dashed")

    ax[0].set_ylabel(
        plot_info["ylabel"],
        fontsize = 20.)

    ax[0].set_title(
        f'{plot_info["title"]}, {this_kinematic_set_title_string}'
        "\n"
        f"(KM15): {km15_cff_string}",
        fontsize = 18.)

    ax[0].legend(fontsize = 18.)

    ax[1].axhline(
        0.,
        linewidth = 0.75,
        linestyle = "dashed")

    ax[1].plot(
        phi_array,
        residual,
        label = "Residuals")

    ax[1].set_xlabel(
        r"$\phi$ [radians]",
        fontsize = 20.)

    ax[1].set_ylabel(
        "Residual",
        fontsize = 18.)

    ax[1].legend(fontsize = 18.)

    fig.tight_layout()

    for extension in ["png", "eps"]:
        fig.savefig(
            plots_directory /
            f'{plot_info["filename"]}_residuals_v{MAJOR_MINOR_NUMBER}.{extension}',
            facecolor = "white"
        )

    plt.close(fig)

### (9.4): Making plots of the *actually-fit observables* during the training process:

#### (9.4.1): Making a plot of *only* the DNN projections *overlaid* on the true observable data:

In [ ]:
for observable_name in enabled_observables:

    print(f"[INFO]: Now making observable plots with {observable_name}")
    
    plot_observable_comparison(
        observable_name = observable_name,
        phi_array = phi_array_in_radians,
        observable_data = OBSERVABLE_DATA,
        statistics_dataframe = replica_statistics_dataframe,
        this_kinematic_set_title_string = this_kinematic_set_title_string,
        km15_cff_string = km15_cff_string,
    )

#### (9.4.2): Making a plot of the DNN projections *and* their residuals:

In [ ]:
for observable_name in enabled_observables:

    print(f"[INFO]: Now making observable plots with {observable_name}")
    
    plot_observable_residuals(
        observable_name = observable_name,
        phi_array = phi_array_in_radians,
        observable_data = OBSERVABLE_DATA,
        statistics_dataframe = replica_statistics_dataframe,
        this_kinematic_set_title_string = this_kinematic_set_title_string,
        km15_cff_string = km15_cff_string,
    )

### (9.5): Making plots

In [ ]:
diagnostic_observables = list(OBSERVABLE_DATA.keys())

for observable_name in diagnostic_observables:

    print(f"[INFO]: Now making observable plots with {observable_name}")

    plot_observable_comparison(
        observable_name = observable_name,
        phi_array = phi_array_in_radians,
        observable_data = OBSERVABLE_DATA,
        statistics_dataframe = replica_statistics_dataframe,
        this_kinematic_set_title_string = this_kinematic_set_title_string,
        km15_cff_string = km15_cff_string,
    )

### (9.6): Plotting CFF Statistical Distributions:

#### (9.6.1): Small function for CFF plot labels:

In [ ]:
def make_cff_plot_label(
    cff_label: str
):
    """
    I am going to customize LaTeX representation of a given CFF
    based on the string version that I received from a datafile.
    """

    # why does this work? because the variable is a STRING
    real_or_imag = cff_label[:2]
    cff_name = cff_label[2:]

    component = { "Re": "Re", "Im": "Im" }[real_or_imag]

    cff_symbol = {
        "H": r"\mathcal{H}",
        "Ht": r"\widetilde{\mathcal{H}}",
        "E": r"\mathcal{E}",
        "Et": r"\widetilde{\mathcal{E}}",
    }[cff_name]

    return rf"{component}$[{cff_symbol}]$"

#### (9.6.1): Actually making the CFF statistical distribution plots:

In [ ]:
_NUMBER_OF_STDDEVS = 4.
_NUMBER_OF_HISTOGRAM_BINS = 30
_NUMBER_OF_GAUSSIAN_POINTS = 200

km15_values = {
    "ReH": CFF_REAL_H_KM15,
    "ImH": CFF_IMAG_H_KM15,
    "ReE": CFF_REAL_E_KM15,
    "ImE": CFF_IMAG_E_KM15,
    "ReHt": CFF_REAL_HT_KM15,
    "ImHt": CFF_IMAG_HT_KM15,
    "ReEt": CFF_REAL_ET_KM15,
    "ImEt": CFF_IMAG_ET_KM15,
}

cff_h_km15 = complex(CFF_REAL_H_KM15, CFF_IMAG_H_KM15)
cff_e_km15 = complex(CFF_REAL_E_KM15, CFF_IMAG_E_KM15)
cff_ht_km15 = complex(CFF_REAL_HT_KM15, CFF_IMAG_HT_KM15)
cff_et_km15 = complex(CFF_REAL_ET_KM15, CFF_IMAG_ET_KM15)

km15_cff_string = (
    rf"$\mathcal{{H}} = {cff_h_km15:.3f}$, "
    rf"$\mathcal{{E}} = {cff_e_km15:.3f}$, "
    rf"$\widetilde{{\mathcal{{H}}}} = {cff_ht_km15:.3f}$, "
    rf"$\widetilde{{\mathcal{{E}}}} = {cff_et_km15:.3f}$"
)

cff_means = []

for cff_label in km15_values:

    cff_name = {
        "ReH": "cff_h_real",
        "ImH": "cff_h_imag",
        "ReHt": "cff_ht_real",
        "ImHt": "cff_ht_imag",
        "ReE": "cff_e_real",
        "ImE": "cff_e_imag",
        "ReEt": "cff_et_real",
        "ImEt": "cff_et_imag",
    }[cff_label]

    is_free = CFF_CONFIG[cff_name][0]
    is_quenched = CFF_CONFIG[cff_name][1]

    if not is_free:
        print(
            f"[INFO]: I found that {cff_label} was not free during the fit. "
            "So, I am not going to generate a histogram for that CFF.")
        continue
        
    corresponding_key = f"{cff_label}_pred"

    if corresponding_key not in replica_cffs_dataframe.columns:
        continue

    cff_prediction_per_replica = replica_cffs_dataframe[corresponding_key]

    # fitting it with a normal distribution
    cff_mean, cff_stddev = norm.fit(cff_prediction_per_replica)
    cff_means.append(cff_mean)
    
    cff_km15_value = km15_values[cff_label]
    
    gaussian_x_values = np.linspace(
        cff_mean - _NUMBER_OF_STDDEVS * cff_stddev,
        cff_mean + _NUMBER_OF_STDDEVS * cff_stddev,
        _NUMBER_OF_GAUSSIAN_POINTS
    )
    
    kinematic_title = (
        rf"$k = {FIXED_K:.3f}$ GeV, "
        rf"$x_B = {FIXED_XB:.3f}$, "
        rf"$t = {FIXED_T:.3f}$ GeV$^2$, "
        rf"$Q^2 = {FIXED_Q_SQUARED:.3f}$ GeV$^2$"
        )
    
    cff_figure, cff_axis = plt.subplots(
        nrows = 1,
        ncols = 1,
        figsize = (10, 10))
    
    cff_axis.hist(
        cff_prediction_per_replica,
        bins = _NUMBER_OF_HISTOGRAM_BINS,
        alpha = 0.6,
        color = "skyblue",
        edgecolor = "black")
    
    cff_axis.plot(
        gaussian_x_values,
        norm.pdf(
            gaussian_x_values,
            cff_mean,
            cff_stddev
        ),
        color = "red",
        linestyle = "--",
        label = (
            fr"Gaussian Fit: $\mu = {cff_mean:.3f}$, $\sigma = {cff_stddev:.3f}$"
        )
    )
    
    cff_axis.axvline(
        cff_km15_value,
        color = "green",
        linestyle = "-",
        linewidth = 2.0,
        label = f"KM15: {cff_km15_value:.3f}")
    
    cff_axis.set_ylabel(
        "Frequency",
        rotation = 90.,
        fontsize = 20.)
    
    cff_axis.set_xlabel(
        make_cff_plot_label(cff_label),
        fontsize = 20.)
    
    cff_axis.set_title(
        rf"{make_cff_plot_label(cff_label)} Distribution, "
        rf"{kinematic_title}"
        "\n"
        rf"(KM15): {km15_cff_string}",
        fontsize = 18.0)
    
    cff_axis.legend(fontsize = 18.0)
    
    cff_axis.text(
        0.00, -0.05,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}",
        transform = cff_axis.transAxes)
    
    cff_figure.tight_layout()
    
    for extension in ["png", "eps"]:
        cff_figure.savefig(
            plots_directory /
            f"cff_{cff_label}_fits_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = "white",
            transparent = False
        )
    
    plt.close(cff_figure)

### (7.4): Construct a Rough Correlation Plot

In [ ]:
CFF_PLOT_INFO = {
    "cff_h_real": {
        "data_column": "ReH_pred",
        "label": r"Re$[\mathcal{H}]$",
        "km15_value": CFF_REAL_H_KM15,
    },
    "cff_h_imag": {
        "data_column": "ImH_pred",
        "label": r"Im$[\mathcal{H}]$",
        "km15_value": CFF_IMAG_H_KM15,
    },
    "cff_ht_real": {
        "data_column": "ReHt_pred",
        "label": r"Re$[\tilde{\mathcal{H}}]$",
        "km15_value": CFF_REAL_HT_KM15,
    },
    "cff_ht_imag": {
        "data_column": "ImHt_pred",
        "label": r"Im$[\tilde{\mathcal{H}}]$",
        "km15_value": CFF_IMAG_HT_KM15,
    },
    "cff_e_real": {
        "data_column": "ReE_pred",
        "label": r"Re$[\mathcal{E}]$",
        "km15_value": CFF_REAL_E_KM15,
    },
    "cff_e_imag": {
        "data_column": "ImE_pred",
        "label": r"Im$[\mathcal{E}]$",
        "km15_value": CFF_IMAG_E_KM15,
    },
    "cff_et_real": {
        "data_column": "ReEt_pred",
        "label": r"Re$[\tilde{\mathcal{E}}]$",
        "km15_value": CFF_REAL_ET_KM15,
    },
    "cff_et_imag": {
        "data_column": "ImEt_pred",
        "label": r"Im$[\tilde{\mathcal{E}}]$",
        "km15_value": CFF_IMAG_ET_KM15,
    },
}

corner_data_columns = [
    CFF_PLOT_INFO[cff_name]["data_column"] for cff_name in FREE_CFF_NAMES
]

corner_labels = [
    CFF_PLOT_INFO[cff_name]["label"] for cff_name in FREE_CFF_NAMES
]

cff_corner_plot_data = np.column_stack([
    replica_cffs_dataframe[column]
    for column in corner_data_columns
])

print(f"[INFO]: Number of free CFFs: {len(FREE_CFF_NAMES)}")
print(f"[INFO]: Corner plot CFFs: {FREE_CFF_NAMES}")
print(f"[INFO]: Shape of the corner plot data: {cff_corner_plot_data.shape}")

cff_corner_plot = corner.corner(
    cff_corner_plot_data,
    labels = corner_labels,
    plot_datapoints = True,
    show_titles = True,
    title_kwargs = {"fontsize": 14},
    label_kwargs = {"fontsize": 14},
    hist_kwargs = {
        'fc': 'skyblue',
        'ec': 'black',
        'histtype': 'bar',
        'lw': 0.9,
        },
    data_kwargs = {
        "alpha": 0.5, 
        "label": "BKM10 Prediction with KM15 CFFs"
        },
    title_fmt = ".3g",
    figsize = (10, 10)
)

for extension in ['png', 'eps']:
    cff_corner_plot.savefig(
        plots_directory /
        f"cff_h_corner_plot_{MAJOR_MINOR_NUMBER}.{extension}",
        facecolor = 'white')